<a href="https://colab.research.google.com/github/Harshal1712/AI-Spend-Intelligence-Platform/blob/main/CEP_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SECTION 1: ENVIRONMENT SETUP
# AGRISENSE:
# An Explainable Ensemble Learning Framework for
# Crop Recommendation and Market Price Prediction
# ============================================================

# ============================================================
# Install Required Libraries
# ============================================================

# Uncomment if running for the first time in Google Colab

!pip install -q xgboost lightgbm shap joblib

# ============================================================
# Import Libraries
# ============================================================

import os
import time
import random
import warnings
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import shap

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

# ============================================================
# Ignore Warning Messages
# ============================================================

warnings.filterwarnings("ignore")

# ============================================================
# Set Random Seed (Reproducibility)
# ============================================================

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ============================================================
# Publication Quality Plot Settings
# ============================================================

sns.set_style("whitegrid")

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 11

# ============================================================
# Create Project Directories
# ============================================================

PROJECT_DIR = "/content/AGRISENSE"

FIGURE_DIR = os.path.join(PROJECT_DIR, "Figures")
MODEL_DIR = os.path.join(PROJECT_DIR, "Saved_Models")
RESULT_DIR = os.path.join(PROJECT_DIR, "Results")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print("Project folders created successfully.")

# ============================================================
# Check GPU Availability
# ============================================================

try:
    import torch

    if torch.cuda.is_available():
        print("=" * 60)
        print("GPU Available")
        print("=" * 60)
        print("GPU Name :", torch.cuda.get_device_name(0))
    else:
        print("=" * 60)
        print("GPU Not Available")
        print("Using CPU")
except ImportError:
    print("=" * 60)
    print("PyTorch not installed. GPU check skipped.")
    print("=" * 60)

# ============================================================
# Display Library Versions
# ============================================================

print("\nLibrary Versions")
print("-" * 40)

print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Matplotlib  :", plt.matplotlib.__version__)
print("Seaborn     :", sns.__version__)
print("SHAP        :", shap.__version__)
print("Joblib      :", joblib.__version__)

import sklearn
print("Scikit-Learn:", sklearn.__version__)

import xgboost
print("XGBoost     :", xgboost.__version__)

import lightgbm
print("LightGBM    :", lightgbm.__version__)

# ============================================================
# Cross Validation Configuration
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("\nCross Validation Strategy")
print(cv)

# ============================================================
# Timer Utility
# ============================================================

def timer(start_time):
    """
    Returns elapsed execution time.
    """
    return round(time.time() - start_time, 2)

# ============================================================
# Save Figure Utility
# ============================================================

def save_figure(filename):
    """
    Save publication-quality figures.
    """
    plt.tight_layout()
    plt.savefig(
        os.path.join(FIGURE_DIR, filename),
        dpi=300,
        bbox_inches="tight"
    )

# ============================================================
# Display Project Structure
# ============================================================

print("\nProject Structure")
print("-" * 40)

print(PROJECT_DIR)
print("│")
print("├── Figures")
print("├── Results")
print("└── Saved_Models")

print("\nEnvironment setup completed successfully.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 2: CROP DATASET LOADING & INITIAL DATA EXPLORATION
# ============================================================

print("=" * 80)
print("SECTION 2 : CROP DATASET LOADING & INITIAL EXPLORATION")
print("=" * 80)

# ============================================================
# Dataset Path
# ============================================================

# Change this path if your dataset is stored elsewhere
DATASET_PATH = "/content/Crop_recommendation.csv"

# ============================================================
# Load Dataset
# ============================================================

crop_df = pd.read_csv(DATASET_PATH)

print("\nDataset loaded successfully.")

# ============================================================
# Display Dataset Shape
# ============================================================

print("\nDataset Shape")
print("-" * 40)
print(f"Rows    : {crop_df.shape[0]}")
print(f"Columns : {crop_df.shape[1]}")

# ============================================================
# Display Column Names
# ============================================================

print("\nColumn Names")
print("-" * 40)

for column in crop_df.columns:
    print(column)

# ============================================================
# Display First Five Rows
# ============================================================

print("\nFirst Five Records")
display(crop_df.head())

# ============================================================
# Display Last Five Rows
# ============================================================

print("\nLast Five Records")
display(crop_df.tail())

# ============================================================
# Dataset Information
# ============================================================

print("\nDataset Information")
print("-" * 40)

crop_df.info()

# ============================================================
# Statistical Summary
# ============================================================

print("\nStatistical Summary")
display(crop_df.describe().T)

# ============================================================
# Missing Value Analysis
# ============================================================

print("\nMissing Values")
print("-" * 40)

missing = crop_df.isnull().sum()

missing_df = pd.DataFrame({
    "Column": missing.index,
    "Missing Values": missing.values,
    "Percentage": round(
        missing.values / len(crop_df) * 100,
        2
    )
})

display(missing_df)

# ============================================================
# Duplicate Records
# ============================================================

duplicates = crop_df.duplicated().sum()

print("\nDuplicate Records")
print("-" * 40)
print(f"Duplicate Rows : {duplicates}")

# ============================================================
# Data Types
# ============================================================

print("\nFeature Data Types")
print("-" * 40)

display(crop_df.dtypes)

# ============================================================
# Target Variable Analysis
# ============================================================

print("\nTarget Variable")
print("-" * 40)

target = "label"

print(f"Target Column : {target}")

print(f"\nTotal Crop Classes : {crop_df[target].nunique()}")

print("\nCrop Names")

print(sorted(crop_df[target].unique()))

# ============================================================
# Class Distribution
# ============================================================

class_distribution = crop_df[target].value_counts().sort_index()

display(class_distribution)

# ============================================================
# Plot Target Distribution
# ============================================================

plt.figure(figsize=(16,6))

sns.countplot(
    data=crop_df,
    x=target,
    order=sorted(crop_df[target].unique()),
    palette="viridis"
)

plt.title(
    "Crop Class Distribution",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Crop Label")

plt.ylabel("Number of Samples")

plt.xticks(rotation=90)

save_figure("Figure_01_Crop_Class_Distribution.png")

plt.show()

# ============================================================
# Dataset Balance Check
# ============================================================

print("\nDataset Balance Statistics")
print("-" * 40)

display(class_distribution.describe())

if class_distribution.std() == 0:
    print("\nDataset is perfectly balanced.")
else:
    print("\nDataset is imbalanced.")

# ============================================================
# Memory Usage
# ============================================================

memory = crop_df.memory_usage(deep=True).sum() / 1024**2

print("\nMemory Usage")
print("-" * 40)

print(f"{memory:.2f} MB")

# ============================================================
# Feature Names
# ============================================================

FEATURES = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

TARGET = "label"

print("\nInput Features")
print("-" * 40)

for feature in FEATURES:
    print(feature)

print(f"\nTarget : {TARGET}")

# ============================================================
# Check Dataset Integrity
# ============================================================

assert TARGET in crop_df.columns

for feature in FEATURES:
    assert feature in crop_df.columns

print("\nDataset Integrity Check Passed.")

# ============================================================
# Section Summary
# ============================================================

print("\n" + "=" * 80)
print("SECTION 2 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"""
Summary
-------

Total Samples           : {crop_df.shape[0]}

Total Features          : {len(FEATURES)}

Target Classes          : {crop_df[TARGET].nunique()}

Missing Values          : {crop_df.isnull().sum().sum()}

Duplicate Records       : {duplicates}

Dataset Memory Usage    : {memory:.2f} MB
""")

In [ ]:
# ============================================================
# SECTION 3 : EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

print("=" * 80)
print("SECTION 3 : EXPLORATORY DATA ANALYSIS")
print("=" * 80)

# ============================================================
# Numerical Features
# ============================================================

numerical_features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

target = "label"

# ============================================================
# 1. Correlation Heatmap
# ============================================================

plt.figure(figsize=(10,8))

corr = crop_df[numerical_features].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="RdYlGn",
    fmt=".2f",
    linewidths=0.5,
    square=True
)

plt.title(
    "Correlation Heatmap of Numerical Features",
    fontsize=16,
    weight="bold"
)

save_figure("Figure_02_Correlation_Heatmap.png")

plt.show()

# ============================================================
# 2. Feature Distribution (Histogram + KDE)
# ============================================================

for feature in numerical_features:

    plt.figure(figsize=(8,5))

    sns.histplot(
        crop_df[feature],
        kde=True,
        bins=30,
        color="steelblue"
    )

    plt.title(
        f"Distribution of {feature}",
        fontsize=15,
        weight="bold"
    )

    plt.xlabel(feature)
    plt.ylabel("Frequency")

    save_figure(f"Distribution_{feature}.png")

    plt.show()

# ============================================================
# 3. Boxplots (Outlier Detection)
# ============================================================

for feature in numerical_features:

    plt.figure(figsize=(8,2.8))

    sns.boxplot(
        x=crop_df[feature],
        color="lightgreen"
    )

    plt.title(
        f"Boxplot of {feature}",
        fontsize=15,
        weight="bold"
    )

    save_figure(f"Boxplot_{feature}.png")

    plt.show()

# ============================================================
# 4. Violin Plots
# ============================================================

for feature in numerical_features:

    plt.figure(figsize=(8,3))

    sns.violinplot(
        x=crop_df[feature],
        color="skyblue"
    )

    plt.title(
        f"Violin Plot of {feature}",
        fontsize=15,
        weight="bold"
    )

    save_figure(f"Violin_{feature}.png")

    plt.show()

# ============================================================
# 5. Pairplot
# ============================================================

sample_df = crop_df.sample(
    min(500, len(crop_df)),
    random_state=42
)

pair = sns.pairplot(
    sample_df,
    vars=numerical_features,
    hue=target,
    corner=True,
    diag_kind="hist"
)

pair.fig.suptitle(
    "Pairwise Relationship Between Features",
    y=1.02,
    fontsize=18
)

pair.savefig(
    os.path.join(
        FIGURE_DIR,
        "Figure_03_Pairplot.png"
    )
)

plt.show()

# ============================================================
# 6. Target Class Distribution
# ============================================================

plt.figure(figsize=(16,6))

order = sorted(crop_df[target].unique())

sns.countplot(
    data=crop_df,
    x=target,
    order=order,
    palette="viridis"
)

plt.xticks(rotation=90)

plt.title(
    "Crop Class Distribution",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Crop")

plt.ylabel("Number of Samples")

save_figure("Figure_04_Target_Distribution.png")

plt.show()

# ============================================================
# 7. Average Feature Values by Crop
# ============================================================

feature_mean = crop_df.groupby(target)[numerical_features].mean()

plt.figure(figsize=(14,10))

sns.heatmap(
    feature_mean,
    cmap="YlGnBu",
    linewidths=0.5
)

plt.title(
    "Average Feature Values for Each Crop",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Features")
plt.ylabel("Crop")

save_figure("Figure_05_Feature_Mean_Heatmap.png")

plt.show()

# ============================================================
# 8. Descriptive Statistics
# ============================================================

print("\nDescriptive Statistics")
print("-" * 60)

display(crop_df[numerical_features].describe().T)

# ============================================================
# 9. Skewness
# ============================================================

print("\nSkewness")
print("-" * 60)

skewness = crop_df[numerical_features].skew()

display(skewness)

# ============================================================
# 10. Kurtosis
# ============================================================

print("\nKurtosis")
print("-" * 60)

kurtosis = crop_df[numerical_features].kurt()

display(kurtosis)

# ============================================================
# 11. Feature-wise Mean, Median and Standard Deviation
# ============================================================

statistics = pd.DataFrame({

    "Mean": crop_df[numerical_features].mean(),

    "Median": crop_df[numerical_features].median(),

    "Std Dev": crop_df[numerical_features].std(),

    "Minimum": crop_df[numerical_features].min(),

    "Maximum": crop_df[numerical_features].max()

})

print("\nFeature Statistics")
print("-" * 60)

display(statistics)

# ============================================================
# 12. Correlation with All Features
# ============================================================

print("\nCorrelation Matrix")
print("-" * 60)

display(corr)

# ============================================================
# 13. Dataset Insights
# ============================================================

print("\nDataset Insights")
print("-" * 60)

print(f"Total Samples        : {len(crop_df)}")
print(f"Total Features       : {len(numerical_features)}")
print(f"Total Crop Classes   : {crop_df[target].nunique()}")

print(f"\nHighest Mean Nitrogen Crop :")

display(
    crop_df.groupby(target)["N"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

print(f"\nHighest Mean Rainfall Crop :")

display(
    crop_df.groupby(target)["rainfall"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

print(f"\nHighest Mean Temperature Crop :")

display(
    crop_df.groupby(target)["temperature"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

print(f"\nHighest Mean pH Crop :")

display(
    crop_df.groupby(target)["ph"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

# ============================================================
# 14. EDA Summary
# ============================================================

print("\n" + "="*80)
print("SECTION 3 COMPLETED SUCCESSFULLY")
print("="*80)

print("""
EDA Completed Successfully

Generated:

✔ Correlation Heatmap

✔ Histogram + KDE

✔ Boxplots

✔ Violin Plots

✔ Pairplot

✔ Crop Distribution

✔ Average Feature Heatmap

✔ Statistical Summary

✔ Skewness

✔ Kurtosis

✔ Feature Statistics

✔ Dataset Insights

✔ Publication Quality Figures Saved
""")

In [ ]:
# ============================================================
# SECTION 4 : CROP DATA PREPROCESSING
# ============================================================

print("=" * 80)
print("SECTION 4 : CROP DATA PREPROCESSING")
print("=" * 80)

# ============================================================
# 1. Check Missing Values
# ============================================================

print("\nChecking Missing Values")
print("-" * 50)

missing_values = crop_df.isnull().sum()

display(missing_values)

if missing_values.sum() == 0:
    print("\nNo Missing Values Found.")
else:
    print("\nMissing Values Detected.")

# ============================================================
# 2. Check Duplicate Records
# ============================================================

print("\nChecking Duplicate Records")
print("-" * 50)

duplicates = crop_df.duplicated().sum()

print("Duplicate Records :", duplicates)

if duplicates > 0:
    crop_df.drop_duplicates(inplace=True)
    print("Duplicate Records Removed.")
else:
    print("No Duplicate Records Found.")

# ============================================================
# 3. Separate Features and Target
# ============================================================

FEATURES = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

TARGET = "label"

X = crop_df[FEATURES]
y = crop_df[TARGET]

print("\nFeatures Selected")
print(FEATURES)

print("\nTarget Variable")
print(TARGET)

# ============================================================
# 4. Encode Target Labels
# ============================================================

print("\nEncoding Crop Labels")
print("-" * 50)

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("\nCrop Labels")

for index, crop in enumerate(label_encoder.classes_):
    print(f"{index} ---> {crop}")

# ============================================================
# 5. Train-Test Split
# ============================================================

print("\nSplitting Dataset")
print("-" * 50)

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_encoded,

    test_size=0.20,

    random_state=RANDOM_STATE,

    stratify=y_encoded

)

print("\nTraining Samples :", X_train.shape[0])

print("Testing Samples  :", X_test.shape[0])

print("Training Features:", X_train.shape)

print("Testing Features :", X_test.shape)

# ============================================================
# 6. Feature Scaling
# ============================================================

print("\nApplying StandardScaler")
print("-" * 50)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling Completed Successfully.")

# ============================================================
# 7. Convert Back to DataFrame
# ============================================================

X_train_scaled = pd.DataFrame(

    X_train_scaled,

    columns=FEATURES

)

X_test_scaled = pd.DataFrame(

    X_test_scaled,

    columns=FEATURES

)

# ============================================================
# 8. Cross Validation Strategy
# ============================================================

print("\nCreating Stratified K-Fold")
print("-" * 50)

cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=RANDOM_STATE

)

print(cv)

# ============================================================
# 9. Dataset Summary
# ============================================================

print("\nDataset Summary")
print("-" * 50)

summary = pd.DataFrame({

    "Dataset":[
        "Training",
        "Testing"
    ],

    "Rows":[
        X_train.shape[0],
        X_test.shape[0]
    ],

    "Columns":[
        X_train.shape[1],
        X_test.shape[1]
    ]

})

display(summary)

# ============================================================
# 10. Display Encoded Classes
# ============================================================

print("\nEncoded Crop Classes")
print("-" * 50)

encoding_df = pd.DataFrame({

    "Encoded Value":range(len(label_encoder.classes_)),

    "Crop":label_encoder.classes_

})

display(encoding_df)

# ============================================================
# 11. Verify Shapes
# ============================================================

print("\nVerification")
print("-" * 50)

print("X Train :", X_train.shape)

print("X Test  :", X_test.shape)

print("y Train :", y_train.shape)

print("y Test  :", y_test.shape)

print("Scaled Train :", X_train_scaled.shape)

print("Scaled Test  :", X_test_scaled.shape)

# ============================================================
# 12. Section Summary
# ============================================================

print("\n" + "=" * 80)
print("SECTION 4 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("""

Preprocessing Completed

✔ Missing Value Check

✔ Duplicate Check

✔ Feature Selection

✔ Label Encoding

✔ Train-Test Split (80:20)

✔ StandardScaler Applied

✔ Stratified 5-Fold Cross Validation Created

Dataset is Ready for Model Training.

""")

In [ ]:
# ============================================================
# SECTION 5.1 : RANDOM FOREST MODEL TRAINING
# ============================================================

print("=" * 80)
print("SECTION 5.1 : RANDOM FOREST CLASSIFIER")
print("=" * 80)

from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# ============================================================
# Start Timer
# ============================================================

start_time = time.time()

# ============================================================
# Define Random Forest Model
# ============================================================

rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# ============================================================
# Hyperparameter Grid
# ============================================================

rf_param_grid = {

    "n_estimators": [100, 200],

    "max_depth": [10, 20, None],

    "min_samples_split": [2, 5],

    "min_samples_leaf": [1, 2],

    "max_features": ["sqrt"]

}

print("\nHyperparameter Grid")
display(rf_param_grid)

# ============================================================
# Grid Search
# ============================================================

rf_grid = GridSearchCV(

    estimator=rf,

    param_grid=rf_param_grid,

    scoring="accuracy",

    cv=cv,

    n_jobs=-1,

    verbose=1

)

rf_grid.fit(X_train, y_train)

# ============================================================
# Best Model
# ============================================================

best_rf = rf_grid.best_estimator_

print("\nBest Parameters")
print("-" * 50)

print(rf_grid.best_params_)

print("\nBest Cross Validation Accuracy")

print(round(rf_grid.best_score_,4))

# ============================================================
# Training Time
# ============================================================

rf_training_time = round(time.time()-start_time,2)

print("\nTraining Time")

print(rf_training_time,"seconds")

# ============================================================
# Prediction
# ============================================================

prediction_start = time.time()

rf_pred = best_rf.predict(X_test)

rf_prediction_time = round(time.time()-prediction_start,4)

# ============================================================
# Performance Metrics
# ============================================================

rf_accuracy = accuracy_score(y_test,rf_pred)

rf_precision = precision_score(
    y_test,
    rf_pred,
    average="weighted"
)

rf_recall = recall_score(
    y_test,
    rf_pred,
    average="weighted"
)

rf_f1 = f1_score(
    y_test,
    rf_pred,
    average="weighted"
)

print("\nModel Performance")
print("-"*50)

print(f"Accuracy  : {rf_accuracy:.4f}")

print(f"Precision : {rf_precision:.4f}")

print(f"Recall    : {rf_recall:.4f}")

print(f"F1 Score  : {rf_f1:.4f}")

print(f"Prediction Time : {rf_prediction_time} sec")

# ============================================================
# Cross Validation Accuracy
# ============================================================

rf_cv = cross_val_score(

    best_rf,

    X_train,

    y_train,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1

)

print("\nCross Validation Scores")

display(rf_cv)

print("Average CV Accuracy :",round(rf_cv.mean(),4))

# ============================================================
# Classification Report
# ============================================================

print("\nClassification Report")
print("-"*50)

print(classification_report(

    y_test,

    rf_pred,

    target_names=label_encoder.classes_

))

# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(

    y_test,

    rf_pred

)

plt.figure(figsize=(12,10))

disp = ConfusionMatrixDisplay(

    confusion_matrix=cm,

    display_labels=label_encoder.classes_

)

disp.plot(

    xticks_rotation=90,

    cmap="Blues",

    values_format="d"

)

plt.title(

    "Random Forest Confusion Matrix",

    fontsize=16,

    weight="bold"

)

save_figure("Figure_06_RF_Confusion_Matrix.png")

plt.show()

# ============================================================
# Feature Importance
# ============================================================

importance = pd.DataFrame({

    "Feature":FEATURES,

    "Importance":best_rf.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

display(importance)

plt.figure(figsize=(8,5))

sns.barplot(

    data=importance,

    x="Importance",

    y="Feature"

)

plt.title(

    "Random Forest Feature Importance",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_07_RF_Feature_Importance.png")

plt.show()

# ============================================================
# Save Model
# ============================================================

joblib.dump(

    best_rf,

    os.path.join(

        MODEL_DIR,

        "RandomForest_Crop_Model.pkl"

    )

)

print("\nRandom Forest Model Saved Successfully.")

# ============================================================
# Store Results
# ============================================================

rf_results = {

    "Model":"Random Forest",

    "Accuracy":rf_accuracy,

    "Precision":rf_precision,

    "Recall":rf_recall,

    "F1 Score":rf_f1,

    "Training Time":rf_training_time,

    "Prediction Time":rf_prediction_time,

    "CV Accuracy":rf_cv.mean()

}

print("\nRandom Forest Results")

display(pd.DataFrame([rf_results]))

print("\n"+"="*80)

print("SECTION 5.1 COMPLETED SUCCESSFULLY")

print("="*80)



# ============================================================
# SECTION 5.2 : XGBOOST MODEL TRAINING
# ============================================================

print("=" * 80)
print("SECTION 5.2 : XGBOOST CLASSIFIER")
print("=" * 80)

from xgboost import XGBClassifier

# ============================================================
# Start Timer
# ============================================================

start_time = time.time()

# ============================================================
# Define XGBoost Model
# ============================================================

xgb = XGBClassifier(

    random_state=RANDOM_STATE,

    objective="multi:softprob",

    eval_metric="mlogloss",

    use_label_encoder=False,

    n_jobs=-1

)

# ============================================================
# Hyperparameter Grid
# ============================================================

xgb_param_grid = {

    "n_estimators":[100,200],

    "max_depth":[4,6,8],

    "learning_rate":[0.01,0.1],

    "subsample":[0.8,1.0],

    "colsample_bytree":[0.8,1.0]

}

print("\nHyperparameter Grid")

display(xgb_param_grid)

# ============================================================
# Grid Search
# ============================================================

xgb_grid = GridSearchCV(

    estimator=xgb,

    param_grid=xgb_param_grid,

    scoring="accuracy",

    cv=cv,

    verbose=1,

    n_jobs=-1

)

xgb_grid.fit(X_train,y_train)

# ============================================================
# Best Model
# ============================================================

best_xgb = xgb_grid.best_estimator_

print("\nBest Parameters")

print(xgb_grid.best_params_)

print("\nBest Cross Validation Accuracy")

print(round(xgb_grid.best_score_,4))

# ============================================================
# Training Time
# ============================================================

xgb_training_time = round(time.time()-start_time,2)

print("\nTraining Time :",xgb_training_time,"seconds")

# ============================================================
# Prediction
# ============================================================

prediction_start = time.time()

xgb_pred = best_xgb.predict(X_test)

xgb_prediction_time = round(time.time()-prediction_start,4)

# ============================================================
# Metrics
# ============================================================

xgb_accuracy = accuracy_score(y_test,xgb_pred)

xgb_precision = precision_score(

    y_test,

    xgb_pred,

    average="weighted"

)

xgb_recall = recall_score(

    y_test,

    xgb_pred,

    average="weighted"

)

xgb_f1 = f1_score(

    y_test,

    xgb_pred,

    average="weighted"

)

print("\nModel Performance")

print("-"*50)

print("Accuracy :",round(xgb_accuracy,4))

print("Precision :",round(xgb_precision,4))

print("Recall :",round(xgb_recall,4))

print("F1 Score :",round(xgb_f1,4))

print("Prediction Time :",xgb_prediction_time,"seconds")

# ============================================================
# Cross Validation
# ============================================================

xgb_cv = cross_val_score(

    best_xgb,

    X_train,

    y_train,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1

)

print("\nCross Validation Scores")

display(xgb_cv)

print("Average CV Accuracy :",round(xgb_cv.mean(),4))

# ============================================================
# Classification Report
# ============================================================

print("\nClassification Report")

print(classification_report(

    y_test,

    xgb_pred,

    target_names=label_encoder.classes_

))

# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(

    y_test,

    xgb_pred

)

plt.figure(figsize=(12,10))

disp = ConfusionMatrixDisplay(

    confusion_matrix=cm,

    display_labels=label_encoder.classes_

)

disp.plot(

    cmap="Greens",

    xticks_rotation=90,

    values_format="d"

)

plt.title(

    "XGBoost Confusion Matrix",

    fontsize=16,

    weight="bold"

)

save_figure("Figure_08_XGBoost_Confusion_Matrix.png")

plt.show()

# ============================================================
# Feature Importance
# ============================================================

importance = pd.DataFrame({

    "Feature":FEATURES,

    "Importance":best_xgb.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

display(importance)

plt.figure(figsize=(8,5))

sns.barplot(

    data=importance,

    x="Importance",

    y="Feature"

)

plt.title(

    "XGBoost Feature Importance",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_09_XGBoost_Feature_Importance.png")

plt.show()

# ============================================================
# Save Model
# ============================================================

joblib.dump(

    best_xgb,

    os.path.join(

        MODEL_DIR,

        "XGBoost_Crop_Model.pkl"

    )

)

print("\nXGBoost Model Saved Successfully.")

# ============================================================
# Store Results
# ============================================================

xgb_results = {

    "Model":"XGBoost",

    "Accuracy":xgb_accuracy,

    "Precision":xgb_precision,

    "Recall":xgb_recall,

    "F1 Score":xgb_f1,

    "Training Time":xgb_training_time,

    "Prediction Time":xgb_prediction_time,

    "CV Accuracy":xgb_cv.mean()

}

display(pd.DataFrame([xgb_results]))

print("\n"+"="*80)

print("SECTION 5.2 COMPLETED SUCCESSFULLY")

print("="*80)



# ============================================================
# SECTION 5.3 : LIGHTGBM MODEL TRAINING
# ============================================================

print("=" * 80)
print("SECTION 5.3 : LIGHTGBM CLASSIFIER")
print("=" * 80)

from lightgbm import LGBMClassifier

# ============================================================
# Start Timer
# ============================================================

start_time = time.time()

# ============================================================
# Define LightGBM Model
# ============================================================

lgbm = LGBMClassifier(
    random_state=RANDOM_STATE,
    verbose=-1
)

# ============================================================
# Hyperparameter Grid
# ============================================================

lgbm_param_grid = {

    "n_estimators":[100,200],

    "learning_rate":[0.01,0.1],

    "max_depth":[5,10,-1],

    "num_leaves":[31,50],

    "subsample":[0.8,1.0],

    "colsample_bytree":[0.8,1.0]

}

print("\nHyperparameter Grid")
display(lgbm_param_grid)

# ============================================================
# Grid Search
# ============================================================

lgbm_grid = GridSearchCV(

    estimator=lgbm,

    param_grid=lgbm_param_grid,

    scoring="accuracy",

    cv=cv,

    verbose=1,

    n_jobs=-1

)

lgbm_grid.fit(X_train,y_train)

# ============================================================
# Best Model
# ============================================================

best_lgbm = lgbm_grid.best_estimator_

print("\nBest Parameters")
print(lgbm_grid.best_params_)

print("\nBest Cross Validation Accuracy")
print(round(lgbm_grid.best_score_,4))

# ============================================================
# Training Time
# ============================================================

lgbm_training_time = round(time.time()-start_time,2)

print("\nTraining Time :",lgbm_training_time,"seconds")

# ============================================================
# Prediction
# ============================================================

prediction_start = time.time()

lgbm_pred = best_lgbm.predict(X_test)

lgbm_prediction_time = round(time.time()-prediction_start,4)

# ============================================================
# Performance Metrics
# ============================================================

lgbm_accuracy = accuracy_score(y_test,lgbm_pred)

lgbm_precision = precision_score(
    y_test,
    lgbm_pred,
    average="weighted"
)

lgbm_recall = recall_score(
    y_test,
    lgbm_pred,
    average="weighted"
)

lgbm_f1 = f1_score(
    y_test,
    lgbm_pred,
    average="weighted"
)

print("\nModel Performance")
print("-"*50)

print("Accuracy :",round(lgbm_accuracy,4))
print("Precision :",round(lgbm_precision,4))
print("Recall :",round(lgbm_recall,4))
print("F1 Score :",round(lgbm_f1,4))
print("Prediction Time :",lgbm_prediction_time,"seconds")

# ============================================================
# Cross Validation
# ============================================================

lgbm_cv = cross_val_score(

    best_lgbm,

    X_train,

    y_train,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1

)

print("\nCross Validation Scores")
display(lgbm_cv)

print("Average CV Accuracy :",round(lgbm_cv.mean(),4))

# ============================================================
# Classification Report
# ============================================================

print("\nClassification Report")

print(classification_report(
    y_test,
    lgbm_pred,
    target_names=label_encoder.classes_
))

# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    lgbm_pred
)

plt.figure(figsize=(12,10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

disp.plot(
    cmap="Purples",
    xticks_rotation=90,
    values_format="d"
)

plt.title(
    "LightGBM Confusion Matrix",
    fontsize=16,
    weight="bold"
)

save_figure("Figure_10_LightGBM_Confusion_Matrix.png")

plt.show()

# ============================================================
# Feature Importance
# ============================================================

importance = pd.DataFrame({

    "Feature":FEATURES,

    "Importance":best_lgbm.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

display(importance)

plt.figure(figsize=(8,5))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title(
    "LightGBM Feature Importance",
    fontsize=15,
    weight="bold"
)

save_figure("Figure_11_LightGBM_Feature_Importance.png")

plt.show()

# ============================================================
# Save Model
# ============================================================

joblib.dump(
    best_lgbm,
    os.path.join(
        MODEL_DIR,
        "LightGBM_Crop_Model.pkl"
    )
)

print("\nLightGBM Model Saved Successfully.")

# ============================================================
# Store Results
# ============================================================

lgbm_results = {

    "Model":"LightGBM",

    "Accuracy":lgbm_accuracy,

    "Precision":lgbm_precision,

    "Recall":lgbm_recall,

    "F1 Score":lgbm_f1,

    "Training Time":lgbm_training_time,

    "Prediction Time":lgbm_prediction_time,

    "CV Accuracy":lgbm_cv.mean()

}

display(pd.DataFrame([lgbm_results]))

print("\n"+"="*80)
print("SECTION 5.3 COMPLETED SUCCESSFULLY")
print("="*80)

In [ ]:
# ============================================================
# SECTION 6 : CROP MODEL COMPARISON
# ============================================================

print("=" * 80)
print("SECTION 6 : CROP MODEL COMPARISON")
print("=" * 80)

# ============================================================
# Create Comparison DataFrame
# ============================================================

comparison_df = pd.DataFrame([

    rf_results,

    xgb_results,

    lgbm_results

])

comparison_df = comparison_df.round(4)

print("\nModel Performance Comparison")
print("-"*80)

display(comparison_df)

# ============================================================
# Save Comparison Table
# ============================================================

comparison_df.to_csv(

    os.path.join(

        RESULT_DIR,

        "Crop_Model_Comparison.csv"

    ),

    index=False

)

# ============================================================
# Accuracy Comparison
# ============================================================

plt.figure(figsize=(8,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="Accuracy",

    palette="viridis"

)

plt.title(

    "Accuracy Comparison",

    fontsize=15,

    weight="bold"

)

plt.ylabel("Accuracy")

save_figure("Figure_12_Accuracy_Comparison.png")

plt.show()

# ============================================================
# Precision Comparison
# ============================================================

plt.figure(figsize=(8,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="Precision",

    palette="magma"

)

plt.title(

    "Precision Comparison",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_13_Precision_Comparison.png")

plt.show()

# ============================================================
# Recall Comparison
# ============================================================

plt.figure(figsize=(8,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="Recall",

    palette="Set2"

)

plt.title(

    "Recall Comparison",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_14_Recall_Comparison.png")

plt.show()

# ============================================================
# F1 Score Comparison
# ============================================================

plt.figure(figsize=(8,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="F1 Score",

    palette="coolwarm"

)

plt.title(

    "F1 Score Comparison",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_15_F1_Comparison.png")

plt.show()

# ============================================================
# Training Time Comparison
# ============================================================

plt.figure(figsize=(8,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="Training Time",

    palette="crest"

)

plt.title(

    "Training Time Comparison",

    fontsize=15,

    weight="bold"

)

plt.ylabel("Time (Seconds)")

save_figure("Figure_16_Training_Time.png")

plt.show()

# ============================================================
# Prediction Time Comparison
# ============================================================

plt.figure(figsize=(8,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="Prediction Time",

    palette="flare"

)

plt.title(

    "Prediction Time Comparison",

    fontsize=15,

    weight="bold"

)

plt.ylabel("Time (Seconds)")

save_figure("Figure_17_Prediction_Time.png")

plt.show()

# ============================================================
# Cross Validation Comparison
# ============================================================

plt.figure(figsize=(8,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="CV Accuracy",

    palette="cubehelix"

)

plt.title(

    "Cross Validation Accuracy",

    fontsize=15,

    weight="bold"

)

plt.ylabel("CV Accuracy")

save_figure("Figure_18_CV_Accuracy.png")

plt.show()

# ============================================================
# Select Best Model
# ============================================================

best_model = comparison_df.loc[
    comparison_df["Accuracy"].idxmax()
]

print("\n"+"="*80)
print("BEST MODEL")
print("="*80)

print(f"Model               : {best_model['Model']}")
print(f"Accuracy            : {best_model['Accuracy']:.4f}")
print(f"Precision           : {best_model['Precision']:.4f}")
print(f"Recall              : {best_model['Recall']:.4f}")
print(f"F1 Score            : {best_model['F1 Score']:.4f}")
print(f"CV Accuracy         : {best_model['CV Accuracy']:.4f}")

# ============================================================
# Ranking
# ============================================================

ranking = comparison_df.sort_values(

    by="Accuracy",

    ascending=False

).reset_index(drop=True)

ranking.index += 1

print("\nModel Ranking")
print("-"*80)

display(ranking)

# ============================================================
# Save Best Model Name
# ============================================================

joblib.dump(

    best_model["Model"],

    os.path.join(

        MODEL_DIR,

        "Best_Crop_Model.pkl"

    )

)

print("\nBest Model Information Saved.")

# ============================================================
# Summary
# ============================================================

print("\n"+"="*80)
print("SECTION 6 COMPLETED SUCCESSFULLY")
print("="*80)

print("""

Completed

✔ Accuracy Comparison

✔ Precision Comparison

✔ Recall Comparison

✔ F1 Comparison

✔ Training Time Comparison

✔ Prediction Time Comparison

✔ Cross Validation Comparison

✔ Best Model Selection

✔ Comparison Table Saved

✔ Publication Quality Figures Saved

""")

In [ ]:
# ============================================================
# SECTION 7 : FEATURE IMPORTANCE ANALYSIS
# ============================================================

print("=" * 80)
print("SECTION 7 : FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

# ============================================================
# Random Forest Feature Importance
# ============================================================

rf_importance = pd.DataFrame({

    "Feature": FEATURES,
    "Random Forest": best_rf.feature_importances_

})

# ============================================================
# XGBoost Feature Importance
# ============================================================

xgb_importance = pd.DataFrame({

    "Feature": FEATURES,
    "XGBoost": best_xgb.feature_importances_

})

# ============================================================
# LightGBM Feature Importance
# ============================================================

lgbm_importance = pd.DataFrame({

    "Feature": FEATURES,
    "LightGBM": best_lgbm.feature_importances_

})

# ============================================================
# Merge All Feature Importances
# ============================================================

importance_df = rf_importance.merge(
    xgb_importance,
    on="Feature"
)

importance_df = importance_df.merge(
    lgbm_importance,
    on="Feature"
)

# ============================================================
# Normalize Values
# ============================================================

importance_df["Random Forest"] = (
    importance_df["Random Forest"] /
    importance_df["Random Forest"].max()
)

importance_df["XGBoost"] = (
    importance_df["XGBoost"] /
    importance_df["XGBoost"].max()
)

importance_df["LightGBM"] = (
    importance_df["LightGBM"] /
    importance_df["LightGBM"].max()
)

# ============================================================
# Average Importance
# ============================================================

importance_df["Average"] = (

    importance_df["Random Forest"] +

    importance_df["XGBoost"] +

    importance_df["LightGBM"]

) / 3

importance_df = importance_df.sort_values(

    by="Average",

    ascending=False

)

print("\nCombined Feature Importance\n")

display(importance_df)

# ============================================================
# Save Table
# ============================================================

importance_df.to_csv(

    os.path.join(

        RESULT_DIR,

        "Combined_Feature_Importance.csv"

    ),

    index=False

)

# ============================================================
# Average Feature Importance Plot
# ============================================================

plt.figure(figsize=(10,6))

sns.barplot(

    data=importance_df,

    x="Average",

    y="Feature",

    palette="viridis"

)

plt.title(

    "Average Feature Importance",

    fontsize=16,

    weight="bold"

)

plt.xlabel("Normalized Importance")

plt.ylabel("Feature")

save_figure("Figure_19_Average_Feature_Importance.png")

plt.show()

# ============================================================
# Comparison Plot
# ============================================================

comparison_plot = importance_df.melt(

    id_vars="Feature",

    value_vars=[

        "Random Forest",

        "XGBoost",

        "LightGBM"

    ],

    var_name="Model",

    value_name="Importance"

)

plt.figure(figsize=(12,6))

sns.barplot(

    data=comparison_plot,

    x="Feature",

    y="Importance",

    hue="Model"

)

plt.title(

    "Feature Importance Comparison Across Models",

    fontsize=16,

    weight="bold"

)

plt.xticks(rotation=45)

plt.legend(title="Model")

save_figure("Figure_20_Feature_Importance_Comparison.png")

plt.show()

# ============================================================
# Heatmap
# ============================================================

heatmap_df = importance_df.set_index("Feature")[

    [

        "Random Forest",

        "XGBoost",

        "LightGBM"

    ]

]

plt.figure(figsize=(8,5))

sns.heatmap(

    heatmap_df,

    annot=True,

    cmap="YlGnBu",

    linewidths=0.5,

    fmt=".2f"

)

plt.title(

    "Feature Importance Heatmap",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_21_Feature_Importance_Heatmap.png")

plt.show()

# ============================================================
# Top 5 Features
# ============================================================

top_features = importance_df.head(5)

print("\nTop 5 Most Important Features")

display(top_features)

# ============================================================
# Save Top Features
# ============================================================

top_features.to_csv(

    os.path.join(

        RESULT_DIR,

        "Top5_Features.csv"

    ),

    index=False

)

# ============================================================
# Research Summary
# ============================================================

print("\n" + "=" * 80)
print("FEATURE IMPORTANCE SUMMARY")
print("=" * 80)

for i, row in top_features.iterrows():

    print(f"{row['Feature']:<15} Average Importance : {row['Average']:.4f}")

print("\nSection 7 Completed Successfully.")


In [ ]:
# ============================================================
# SECTION 8 : SHAP EXPLAINABILITY
# ============================================================

print("=" * 80)
print("SECTION 8 : SHAP EXPLAINABILITY")
print("=" * 80)

import shap

# ============================================================
# Select Best Model
# ============================================================

model_dict = {
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgbm
}

best_model_name = comparison_df.loc[
    comparison_df["Accuracy"].idxmax(),
    "Model"
]

best_model = model_dict[best_model_name]

print(f"\nBest Model Selected : {best_model_name}")

# ============================================================
# Create SHAP Explainer
# ============================================================

explainer = shap.TreeExplainer(best_model)

print("\nGenerating SHAP Values...")

shap_values = explainer.shap_values(X_test)

print("SHAP Values Generated Successfully.")

# ============================================================
# SHAP Summary Plot
# ============================================================

print("\nGenerating SHAP Summary Plot...")

plt.figure()

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURES,
    show=False
)

save_figure("Figure_22_SHAP_Summary.png")

plt.show()

# ============================================================
# SHAP Bar Plot
# ============================================================

print("\nGenerating SHAP Feature Importance Plot...")

plt.figure()

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURES,
    plot_type="bar",
    show=False
)

save_figure("Figure_23_SHAP_Bar.png")

plt.show()

# ============================================================
# SHAP Dependence Plot
# ============================================================

important_feature = importance_df.iloc[0]["Feature"]

print(f"\nMost Important Feature : {important_feature}")

plt.figure()

shap.dependence_plot(
    important_feature,
    shap_values,
    X_test,
    feature_names=FEATURES,
    show=False
)

save_figure("Figure_24_SHAP_Dependence.png")

plt.show()

# ============================================================
# Waterfall Plot
# ============================================================

print("\nGenerating Waterfall Plot...")

sample_index = 0

explanation = shap.Explanation(

    values=shap_values[sample_index],

    base_values=explainer.expected_value,

    data=X_test.iloc[sample_index],

    feature_names=FEATURES

)

plt.figure()

shap.plots.waterfall(
    explanation,
    show=False
)

save_figure("Figure_25_SHAP_Waterfall.png")

plt.show()

# ============================================================
# Force Plot
# ============================================================

print("\nGenerating Force Plot...")

force_plot = shap.force_plot(

    explainer.expected_value,

    shap_values[sample_index],

    X_test.iloc[sample_index],

    feature_names=FEATURES,

    matplotlib=True,

    show=False

)

save_figure("Figure_26_SHAP_Force.png")

plt.show()

# ============================================================
# Explain One Prediction
# ============================================================

predicted_label = best_model.predict(
    X_test.iloc[[sample_index]]
)[0]

predicted_crop = label_encoder.inverse_transform(
    [predicted_label]
)[0]

print("\n" + "=" * 80)
print("LOCAL PREDICTION EXPLANATION")
print("=" * 80)

print(f"Predicted Crop : {predicted_crop}")

print("\nInput Features")

display(
    X_test.iloc[[sample_index]]
)

# ============================================================
# SHAP Importance Table
# ============================================================

if isinstance(shap_values, list):

    mean_shap = np.mean(
        np.abs(shap_values),
        axis=(0,1)
    )

else:

    mean_shap = np.abs(
        shap_values
    ).mean(axis=0)

shap_df = pd.DataFrame({

    "Feature": FEATURES,

    "Mean |SHAP|": mean_shap

})

shap_df = shap_df.sort_values(

    by="Mean |SHAP|",

    ascending=False

)

print("\nMean SHAP Importance")

display(shap_df)

shap_df.to_csv(

    os.path.join(

        RESULT_DIR,

        "SHAP_Feature_Importance.csv"

    ),

    index=False

)

# ============================================================
# Completion Message
# ============================================================

print("\n" + "=" * 80)
print("SECTION 8 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("""

Completed

✔ SHAP Summary Plot

✔ SHAP Bar Plot

✔ SHAP Dependence Plot

✔ SHAP Waterfall Plot

✔ SHAP Force Plot

✔ Global Explainability

✔ Local Explainability

✔ SHAP Importance Table

✔ Publication Quality Figures Saved

""")

In [ ]:
# ============================================================
# PHASE 2 : MARKET PRICE PREDICTION
# SECTION 9 : PRICE DATASET LOADING
# ============================================================

print("=" * 80)
print("PHASE 2 : MARKET PRICE PREDICTION")
print("SECTION 9 : PRICE DATASET LOADING")
print("=" * 80)

# ============================================================
# Load Dataset
# ============================================================

PRICE_DATASET_PATH = "/content/final_price_dataset.csv"

price_df = pd.read_csv(PRICE_DATASET_PATH)

print("\n✅ Dataset Loaded Successfully!")

# ============================================================
# Remove Unnecessary Column
# ============================================================

if "date" in price_df.columns:
    price_df.drop(columns=["date"], inplace=True)
    print("✅ 'date' column removed.")

# ============================================================
# Dataset Shape
# ============================================================

print("\nDataset Shape")
print("-" * 40)
print(f"Rows    : {price_df.shape[0]:,}")
print(f"Columns : {price_df.shape[1]}")

# ============================================================
# Display Dataset
# ============================================================

print("\nFirst Five Records")
display(price_df.head())

print("\nLast Five Records")
display(price_df.tail())

# ============================================================
# Dataset Information
# ============================================================

print("\nDataset Information")
print("-" * 40)

price_df.info()

# ============================================================
# Data Types
# ============================================================

dtype_df = pd.DataFrame({

    "Column": price_df.columns,

    "Data Type": price_df.dtypes.astype(str)

})

print("\nData Types")

display(dtype_df)

# ============================================================
# Missing Values
# ============================================================

missing_df = pd.DataFrame({

    "Missing Values": price_df.isnull().sum(),

    "Percentage": (
        price_df.isnull().sum() /
        len(price_df)
    ) * 100

})

print("\nMissing Values")

display(missing_df)

# ============================================================
# Duplicate Values
# ============================================================

duplicates = price_df.duplicated().sum()

print("\nDuplicate Records")
print("-" * 40)
print(f"Duplicates : {duplicates}")

# ============================================================
# Descriptive Statistics
# ============================================================

print("\nNumerical Statistics")

display(price_df.describe())

print("\nCategorical Statistics")

display(price_df.describe(include="object"))

# ============================================================
# Memory Usage
# ============================================================

memory_usage = price_df.memory_usage(deep=True).sum() / (1024 ** 2)

print("\nMemory Usage")
print("-" * 40)
print(f"{memory_usage:.2f} MB")

# ============================================================
# Target Variable
# ============================================================

TARGET = "modal_price"

print("\nTarget Variable")
print("-" * 40)
print(TARGET)

# ============================================================
# Feature Columns
# ============================================================

FEATURES = price_df.drop(columns=[TARGET]).columns.tolist()

print("\nFeature Columns")

for feature in FEATURES:
    print(f"• {feature}")

# ============================================================
# Numerical & Categorical Features
# ============================================================

categorical_features = price_df.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = price_df.select_dtypes(
    exclude=["object"]
).columns.tolist()

if TARGET in numerical_features:
    numerical_features.remove(TARGET)

print("\nCategorical Features")
print(categorical_features)

print("\nNumerical Features")
print(numerical_features)

# ============================================================
# Target Statistics
# ============================================================

print("\nTarget Statistics")
print("-" * 40)

display(price_df[TARGET].describe())

# ============================================================
# Unique Values
# ============================================================

unique_df = pd.DataFrame({

    "Feature": price_df.columns,

    "Unique Values": [

        price_df[col].nunique()

        for col in price_df.columns

    ]

})

print("\nUnique Values Per Column")

display(unique_df)

# ============================================================
# Initial Observations
# ============================================================

print("\n" + "=" * 80)
print("INITIAL OBSERVATIONS")
print("=" * 80)

print(f"""
Total Records        : {price_df.shape[0]:,}
Total Features       : {len(FEATURES)}
Target Variable      : {TARGET}

Categorical Features : {len(categorical_features)}
Numerical Features   : {len(numerical_features)}

Duplicate Records    : {duplicates}

Missing Values       : {price_df.isnull().sum().sum()}
""")

print("\n✅ Section 9 Completed Successfully.")

In [ ]:
# ============================================================
# SECTION 10 : EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

print("=" * 80)
print("SECTION 10 : EXPLORATORY DATA ANALYSIS")
print("=" * 80)

# ============================================================
# Figure Style
# ============================================================

plt.style.use("ggplot")

# ============================================================
# 1. Target Variable Distribution
# ============================================================

plt.figure(figsize=(10,6))

sns.histplot(
    price_df[TARGET],
    bins=40,
    kde=True,
    color="royalblue"
)

plt.title(
    "Distribution of Modal Price",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Modal Price")
plt.ylabel("Frequency")

save_figure("Figure_27_Target_Distribution.png")

plt.show()

# ============================================================
# 2. Commodity Distribution
# ============================================================

plt.figure(figsize=(14,8))

price_df["Commodity"].value_counts().plot(
    kind="bar"
)

plt.title(
    "Commodity Distribution",
    fontsize=16,
    weight="bold"
)

plt.ylabel("Count")

plt.xticks(rotation=90)

save_figure("Figure_28_Commodity_Distribution.png")

plt.show()

# ============================================================
# 3. State Distribution
# ============================================================

plt.figure(figsize=(8,5))

sns.countplot(
    data=price_df,
    x="state_name"
)

plt.title(
    "State Distribution",
    fontsize=16,
    weight="bold"
)

plt.xticks(rotation=45)

save_figure("Figure_29_State_Distribution.png")

plt.show()

# ============================================================
# 4. Top 15 Districts
# ============================================================

top_districts = price_df["district_name"].value_counts().head(15)

plt.figure(figsize=(12,6))

sns.barplot(
    x=top_districts.values,
    y=top_districts.index
)

plt.title(
    "Top 15 Districts by Records",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Count")

save_figure("Figure_30_Top_Districts.png")

plt.show()

# ============================================================
# 5. Monthly Trend
# ============================================================

monthly_price = price_df.groupby("Month")[TARGET].mean()

plt.figure(figsize=(10,5))

plt.plot(
    monthly_price.index,
    monthly_price.values,
    marker="o",
    linewidth=2
)

plt.title(
    "Average Monthly Modal Price",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Month")
plt.ylabel("Average Price")

save_figure("Figure_31_Monthly_Trend.png")

plt.show()

# ============================================================
# 6. Yearly Trend
# ============================================================

yearly_price = price_df.groupby("Year")[TARGET].mean()

plt.figure(figsize=(10,5))

plt.plot(
    yearly_price.index,
    yearly_price.values,
    marker="o",
    linewidth=2
)

plt.title(
    "Average Yearly Modal Price",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Year")
plt.ylabel("Average Price")

save_figure("Figure_32_Yearly_Trend.png")

plt.show()

# ============================================================
# 7. Correlation Heatmap
# ============================================================

plt.figure(figsize=(8,6))

corr = price_df[numerical_features + [TARGET]].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title(
    "Correlation Heatmap",
    fontsize=16,
    weight="bold"
)

save_figure("Figure_33_Correlation_Heatmap.png")

plt.show()

# ============================================================
# 8. Histograms for Numerical Features
# ============================================================

price_df[numerical_features].hist(
    figsize=(14,10),
    bins=30
)

plt.tight_layout()

save_figure("Figure_34_Numerical_Histograms.png")

plt.show()

# ============================================================
# 9. Boxplots for Numerical Features
# ============================================================

for feature in numerical_features:

    plt.figure(figsize=(8,4))

    sns.boxplot(
        x=price_df[feature]
    )

    plt.title(
        f"{feature} Boxplot",
        fontsize=14,
        weight="bold"
    )

    save_figure(f"Boxplot_{feature}.png")

    plt.show()

# ============================================================
# 10. Arrivals vs Modal Price
# ============================================================

plt.figure(figsize=(8,6))

sns.scatterplot(
    data=price_df,
    x="arrivals_in_qtl",
    y=TARGET,
    alpha=0.5
)

plt.title(
    "Arrivals vs Modal Price",
    fontsize=16,
    weight="bold"
)

save_figure("Figure_35_Arrivals_vs_Price.png")

plt.show()

# ============================================================
# 11. Top Commodities by Average Price
# ============================================================

top_price = (
    price_df.groupby("Commodity")[TARGET]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12,6))

sns.barplot(
    x=top_price.values,
    y=top_price.index
)

plt.title(
    "Top 15 Commodities by Average Price",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Average Modal Price")

save_figure("Figure_36_Top_Commodities.png")

plt.show()

# ============================================================
# 12. Top Districts by Average Price
# ============================================================

district_price = (
    price_df.groupby("district_name")[TARGET]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12,6))

sns.barplot(
    x=district_price.values,
    y=district_price.index
)

plt.title(
    "Top 15 Districts by Average Price",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Average Modal Price")

save_figure("Figure_37_District_Price.png")

plt.show()

# ============================================================
# 13. Pairplot (Sampled Dataset)
# ============================================================

sample_df = price_df.sample(
    min(500, len(price_df)),
    random_state=RANDOM_STATE
)

sns.pairplot(
    sample_df[
        numerical_features + [TARGET]
    ]
)

save_figure("Figure_38_Pairplot.png")

plt.show()

# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 80)
print("EDA COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"""
Dataset Records     : {len(price_df):,}
Features            : {len(FEATURES)}
Target              : {TARGET}

Categorical         : {len(categorical_features)}
Numerical           : {len(numerical_features)}

Publication Figures : 12+

EDA Completed Successfully.
""")

In [ ]:
# ============================================================
# SECTION 11 : DATA PREPROCESSING
# ============================================================

print("=" * 80)
print("SECTION 11 : DATA PREPROCESSING")
print("=" * 80)

# ============================================================
# Dataset Copy
# ============================================================

df = price_df.copy()

print("\nWorking Dataset Created Successfully")

# ============================================================
# Missing Values
# ============================================================

print("\nChecking Missing Values")

missing = df.isnull().sum()

display(missing)

if missing.sum() == 0:
    print("✅ No Missing Values Found")
else:
    print("Missing Values Detected")
    df.dropna(inplace=True)

# ============================================================
# Duplicate Records
# ============================================================

duplicates = df.duplicated().sum()

print("\nDuplicate Records :", duplicates)

if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates Removed Successfully")
else:
    print("No Duplicate Records Found")

# ============================================================
# Dataset Shape
# ============================================================

print("\nDataset Shape After Cleaning")

print(df.shape)

# ============================================================
# Feature & Target Selection
# ============================================================

TARGET = "modal_price"

price_features = [

    "APMC",

    "Commodity",

    "Year",

    "Month",

    "arrivals_in_qtl",

    "min_price",

    "max_price",

    "district_name",

    "state_name"

]

X = df[price_features]

y = df[TARGET]

print("\nFeatures Selected")

display(X.head())

print("\nTarget Variable")

display(y.head())

# ============================================================
# Label Encoding
# ============================================================

print("\nEncoding Categorical Features")

from sklearn.preprocessing import LabelEncoder

label_encoders = {}

categorical_columns = [

    "APMC",

    "Commodity",

    "district_name",

    "state_name"

]

for column in categorical_columns:

    encoder = LabelEncoder()

    X[column] = encoder.fit_transform(X[column])

    label_encoders[column] = encoder

    print(f"{column} Encoded Successfully")

# ============================================================
# Save Encoders
# ============================================================

for column, encoder in label_encoders.items():

    joblib.dump(

        encoder,

        os.path.join(

            MODEL_DIR,

            f"{column}_Encoder.pkl"

        )

    )

print("\nAll Encoders Saved Successfully")

# ============================================================
# Train Test Split
# ============================================================

print("\nCreating Training & Testing Dataset")

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=RANDOM_STATE

)

print("Training Shape :", X_train.shape)

print("Testing Shape  :", X_test.shape)

# ============================================================
# Dataset Verification
# ============================================================

print("\nTraining Features")

display(X_train.head())

print("\nTraining Target")

display(y_train.head())

print("\nTesting Features")

display(X_test.head())

print("\nTesting Target")

display(y_test.head())

# ============================================================
# Feature Information
# ============================================================

feature_info = pd.DataFrame({

    "Feature": X.columns,

    "Data Type": X.dtypes.astype(str),

    "Unique Values": [X[col].nunique() for col in X.columns]

})

print("\nFeature Summary")

display(feature_info)

# ============================================================
# Correlation with Target
# ============================================================

correlation = pd.concat(

    [X, y],

    axis=1

).corr()["modal_price"].sort_values(

    ascending=False

)

print("\nCorrelation with Target")

display(correlation)

# ============================================================
# Save Processed Dataset
# ============================================================

processed_df = pd.concat(

    [X, y],

    axis=1

)

processed_df.to_csv(

    os.path.join(

        RESULT_DIR,

        "Processed_Price_Dataset.csv"

    ),

    index=False

)

print("\nProcessed Dataset Saved Successfully")

# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 80)
print("SECTION 11 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"""

Training Samples : {len(X_train)}

Testing Samples  : {len(X_test)}

Total Features   : {len(price_features)}

Target Variable  : {TARGET}

Encoders Saved   : {len(label_encoders)}

Dataset Ready For Regression Models

""")

In [ ]:
# ============================================================
# SECTION 12.1 : RANDOM FOREST REGRESSOR
# ============================================================

print("=" * 80)
print("SECTION 12.1 : RANDOM FOREST REGRESSOR")
print("=" * 80)

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

# ============================================================
# Start Timer
# ============================================================

start_time = time.time()

# ============================================================
# Model
# ============================================================

rf_regressor = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# ============================================================
# Hyperparameter Grid
# ============================================================

rf_param_grid = {

    "n_estimators": [100, 200],

    "max_depth": [10, 20, None],

    "min_samples_split": [2, 5],

    "min_samples_leaf": [1, 2]

}

print("\nHyperparameter Grid")
display(rf_param_grid)

# ============================================================
# Grid Search
# ============================================================

rf_grid = GridSearchCV(

    estimator=rf_regressor,

    param_grid=rf_param_grid,

    cv=5,

    scoring="r2",

    n_jobs=-1,

    verbose=1

)

rf_grid.fit(X_train, y_train)

# ============================================================
# Best Model
# ============================================================

best_rf_regressor = rf_grid.best_estimator_

print("\nBest Parameters")
print(rf_grid.best_params_)

print("\nBest Cross Validation R²")
print(round(rf_grid.best_score_, 4))

# ============================================================
# Training Time
# ============================================================

rf_training_time = round(time.time() - start_time, 2)

print("\nTraining Time :", rf_training_time, "seconds")

# ============================================================
# Prediction
# ============================================================

prediction_start = time.time()

rf_predictions = best_rf_regressor.predict(X_test)

rf_prediction_time = round(
    time.time() - prediction_start,
    4
)

# ============================================================
# Evaluation Metrics
# ============================================================

rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_mse = mean_squared_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(rf_mse)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

rf_mape = mean_absolute_percentage_error(
    y_test,
    rf_predictions
)

print("\nModel Performance")
print("-" * 50)

print(f"MAE  : {rf_mae:.2f}")
print(f"MSE  : {rf_mse:.2f}")
print(f"RMSE : {rf_rmse:.2f}")
print(f"R²   : {rf_r2:.4f}")
print(f"MAPE : {rf_mape:.4f}")

# ============================================================
# Cross Validation
# ============================================================

rf_cv_scores = cross_val_score(

    best_rf_regressor,

    X_train,

    y_train,

    cv=5,

    scoring="r2",

    n_jobs=-1

)

print("\nCross Validation Scores")
display(rf_cv_scores)

print(
    "Average CV R² :",
    round(rf_cv_scores.mean(), 4)
)

# ============================================================
# Actual vs Predicted
# ============================================================

plt.figure(figsize=(7,7))

plt.scatter(
    y_test,
    rf_predictions,
    alpha=0.6
)

plt.plot(

    [y_test.min(), y_test.max()],

    [y_test.min(), y_test.max()],

    color="red",

    linestyle="--"

)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")

plt.title(
    "Random Forest : Actual vs Predicted",
    fontsize=15,
    weight="bold"
)

save_figure("Figure_39_RF_Actual_vs_Predicted.png")

plt.show()

# ============================================================
# Residual Plot
# ============================================================

residuals = y_test - rf_predictions

plt.figure(figsize=(8,5))

sns.scatterplot(

    x=rf_predictions,

    y=residuals

)

plt.axhline(

    0,

    color="red",

    linestyle="--"

)

plt.xlabel("Predicted Price")
plt.ylabel("Residual")

plt.title(
    "Residual Plot",
    fontsize=15,
    weight="bold"
)

save_figure("Figure_40_RF_Residuals.png")

plt.show()

# ============================================================
# Feature Importance
# ============================================================

importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance": best_rf_regressor.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

display(importance)

plt.figure(figsize=(8,5))

sns.barplot(

    data=importance,

    x="Importance",

    y="Feature"

)

plt.title(
    "Random Forest Feature Importance",
    fontsize=15,
    weight="bold"
)

save_figure("Figure_41_RF_Feature_Importance.png")

plt.show()

# ============================================================
# Save Model
# ============================================================

joblib.dump(

    best_rf_regressor,

    os.path.join(

        MODEL_DIR,

        "RandomForest_Price_Model.pkl"

    )

)

print("\nRandom Forest Model Saved Successfully.")

# ============================================================
# Store Results
# ============================================================

rf_regression_results = {

    "Model": "Random Forest",

    "MAE": rf_mae,

    "MSE": rf_mse,

    "RMSE": rf_rmse,

    "R2": rf_r2,

    "MAPE": rf_mape,

    "Training Time": rf_training_time,

    "Prediction Time": rf_prediction_time,

    "CV R2": rf_cv_scores.mean()

}

display(pd.DataFrame([rf_regression_results]))

print("\n" + "=" * 80)
print("SECTION 12.1 COMPLETED SUCCESSFULLY")
print("=" * 80)


# ============================================================
# SECTION 12.2 : XGBOOST REGRESSOR
# ============================================================

print("=" * 80)
print("SECTION 12.2 : XGBOOST REGRESSOR")
print("=" * 80)

from xgboost import XGBRegressor

# ============================================================
# Start Timer
# ============================================================

start_time = time.time()

# ============================================================
# Define Model
# ============================================================

xgb_regressor = XGBRegressor(

    objective="reg:squarederror",

    random_state=RANDOM_STATE,

    n_jobs=-1

)

# ============================================================
# Hyperparameter Grid
# ============================================================

xgb_param_grid = {

    "n_estimators": [100, 200],

    "learning_rate": [0.01, 0.1],

    "max_depth": [4, 6, 8],

    "subsample": [0.8, 1.0],

    "colsample_bytree": [0.8, 1.0]

}

print("\nHyperparameter Grid")

display(xgb_param_grid)

# ============================================================
# Grid Search
# ============================================================

xgb_grid = GridSearchCV(

    estimator=xgb_regressor,

    param_grid=xgb_param_grid,

    scoring="r2",

    cv=5,

    n_jobs=-1,

    verbose=1

)

xgb_grid.fit(X_train, y_train)

# ============================================================
# Best Model
# ============================================================

best_xgb_regressor = xgb_grid.best_estimator_

print("\nBest Parameters")

print(xgb_grid.best_params_)

print("\nBest Cross Validation R²")

print(round(xgb_grid.best_score_, 4))

# ============================================================
# Training Time
# ============================================================

xgb_training_time = round(

    time.time() - start_time,

    2

)

print("\nTraining Time :", xgb_training_time, "seconds")

# ============================================================
# Prediction
# ============================================================

prediction_start = time.time()

xgb_predictions = best_xgb_regressor.predict(X_test)

xgb_prediction_time = round(

    time.time() - prediction_start,

    4

)

# ============================================================
# Evaluation Metrics
# ============================================================

xgb_mae = mean_absolute_error(

    y_test,

    xgb_predictions

)

xgb_mse = mean_squared_error(

    y_test,

    xgb_predictions

)

xgb_rmse = np.sqrt(xgb_mse)

xgb_r2 = r2_score(

    y_test,

    xgb_predictions

)

xgb_mape = mean_absolute_percentage_error(

    y_test,

    xgb_predictions

)

print("\nModel Performance")
print("-" * 50)

print(f"MAE              : {xgb_mae:.2f}")
print(f"MSE              : {xgb_mse:.2f}")
print(f"RMSE             : {xgb_rmse:.2f}")
print(f"R² Score         : {xgb_r2:.4f}")
print(f"MAPE             : {xgb_mape:.4f}")
print(f"Prediction Time  : {xgb_prediction_time} seconds")

# ============================================================
# Cross Validation
# ============================================================

xgb_cv_scores = cross_val_score(

    best_xgb_regressor,

    X_train,

    y_train,

    cv=5,

    scoring="r2",

    n_jobs=-1

)

print("\nCross Validation Scores")

display(xgb_cv_scores)

print("Average CV R² :", round(xgb_cv_scores.mean(), 4))

# ============================================================
# Actual vs Predicted
# ============================================================

plt.figure(figsize=(7,7))

plt.scatter(

    y_test,

    xgb_predictions,

    alpha=0.6

)

plt.plot(

    [y_test.min(), y_test.max()],

    [y_test.min(), y_test.max()],

    "r--",

    linewidth=2

)

plt.xlabel("Actual Price")

plt.ylabel("Predicted Price")

plt.title(

    "XGBoost : Actual vs Predicted",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_42_XGB_Actual_vs_Predicted.png")

plt.show()

# ============================================================
# Residual Plot
# ============================================================

residuals = y_test - xgb_predictions

plt.figure(figsize=(8,5))

sns.scatterplot(

    x=xgb_predictions,

    y=residuals

)

plt.axhline(

    y=0,

    color="red",

    linestyle="--"

)

plt.xlabel("Predicted Price")

plt.ylabel("Residual")

plt.title(

    "XGBoost Residual Plot",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_43_XGB_Residuals.png")

plt.show()

# ============================================================
# Feature Importance
# ============================================================

importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance": best_xgb_regressor.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

display(importance)

plt.figure(figsize=(8,5))

sns.barplot(

    data=importance,

    x="Importance",

    y="Feature"

)

plt.title(

    "XGBoost Feature Importance",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_44_XGB_Feature_Importance.png")

plt.show()

# ============================================================
# Save Model
# ============================================================

joblib.dump(

    best_xgb_regressor,

    os.path.join(

        MODEL_DIR,

        "XGBoost_Price_Model.pkl"

    )

)

print("\nXGBoost Model Saved Successfully.")

# ============================================================
# Store Results
# ============================================================

xgb_regression_results = {

    "Model": "XGBoost",

    "MAE": xgb_mae,

    "MSE": xgb_mse,

    "RMSE": xgb_rmse,

    "R2": xgb_r2,

    "MAPE": xgb_mape,

    "Training Time": xgb_training_time,

    "Prediction Time": xgb_prediction_time,

    "CV R2": xgb_cv_scores.mean()

}

display(pd.DataFrame([xgb_regression_results]))

print("\n" + "=" * 80)
print("SECTION 12.2 COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
# ============================================================
# SECTION 13 : REGRESSION MODEL COMPARISON
# ============================================================

print("=" * 80)
print("SECTION 13 : REGRESSION MODEL COMPARISON")
print("=" * 80)

# ============================================================
# Create Comparison DataFrame
# ============================================================

comparison_df = pd.DataFrame([

    rf_regression_results,

    xgb_regression_results

])

comparison_df = comparison_df.round(4)

print("\nModel Comparison")

display(comparison_df)

# ============================================================
# Save Comparison Table
# ============================================================

comparison_df.to_csv(

    os.path.join(

        RESULT_DIR,

        "Regression_Model_Comparison.csv"

    ),

    index=False

)

# ============================================================
# MAE Comparison
# ============================================================

plt.figure(figsize=(7,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="MAE",

    palette="viridis"

)

plt.title(

    "Mean Absolute Error Comparison",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_45_MAE_Comparison.png")

plt.show()

# ============================================================
# RMSE Comparison
# ============================================================

plt.figure(figsize=(7,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="RMSE",

    palette="magma"

)

plt.title(

    "RMSE Comparison",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_46_RMSE_Comparison.png")

plt.show()

# ============================================================
# R² Comparison
# ============================================================

plt.figure(figsize=(7,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="R2",

    palette="crest"

)

plt.title(

    "R² Score Comparison",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_47_R2_Comparison.png")

plt.show()

# ============================================================
# MAPE Comparison
# ============================================================

plt.figure(figsize=(7,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="MAPE",

    palette="flare"

)

plt.title(

    "MAPE Comparison",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_48_MAPE_Comparison.png")

plt.show()

# ============================================================
# Training Time Comparison
# ============================================================

plt.figure(figsize=(7,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="Training Time",

    palette="Blues"

)

plt.title(

    "Training Time Comparison",

    fontsize=15,

    weight="bold"

)

plt.ylabel("Seconds")

save_figure("Figure_49_Training_Time.png")

plt.show()

# ============================================================
# Prediction Time Comparison
# ============================================================

plt.figure(figsize=(7,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="Prediction Time",

    palette="Greens"

)

plt.title(

    "Prediction Time Comparison",

    fontsize=15,

    weight="bold"

)

plt.ylabel("Seconds")

save_figure("Figure_50_Prediction_Time.png")

plt.show()

# ============================================================
# Cross Validation Comparison
# ============================================================

plt.figure(figsize=(7,5))

sns.barplot(

    data=comparison_df,

    x="Model",

    y="CV R2",

    palette="Purples"

)

plt.title(

    "Cross Validation R² Comparison",

    fontsize=15,

    weight="bold"

)

save_figure("Figure_51_CV_R2.png")

plt.show()

# ============================================================
# Select Best Model
# ============================================================

best_model = comparison_df.loc[
    comparison_df["R2"].idxmax()
]

print("\n" + "=" * 80)
print("BEST REGRESSION MODEL")
print("=" * 80)

print(f"Model            : {best_model['Model']}")
print(f"R² Score         : {best_model['R2']:.4f}")
print(f"RMSE             : {best_model['RMSE']:.2f}")
print(f"MAE              : {best_model['MAE']:.2f}")
print(f"MAPE             : {best_model['MAPE']:.4f}")
print(f"CV R²            : {best_model['CV R2']:.4f}")

# ============================================================
# Model Ranking
# ============================================================

ranking = comparison_df.sort_values(

    by="R2",

    ascending=False

).reset_index(drop=True)

ranking.index += 1

print("\nModel Ranking")

display(ranking)

# ============================================================
# Save Best Model Name
# ============================================================

joblib.dump(

    best_model["Model"],

    os.path.join(

        MODEL_DIR,

        "Best_Price_Model.pkl"

    )

)

print("\nBest Model Information Saved Successfully.")

# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 80)
print("SECTION 13 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("""

Completed

✔ MAE Comparison

✔ RMSE Comparison

✔ R² Comparison

✔ MAPE Comparison

✔ Training Time Comparison

✔ Prediction Time Comparison

✔ Cross Validation Comparison

✔ Best Model Selection

✔ Model Ranking

✔ Comparison Table Saved

✔ Publication Quality Figures Saved

""")

In [ ]:
# ============================================================
# SECTION 14 : SHAP EXPLAINABILITY
# ============================================================

print("=" * 80)
print("SECTION 14 : SHAP EXPLAINABILITY")
print("=" * 80)

import shap

# ============================================================
# Select Best Model
# ============================================================

model_dict = {
    "Random Forest": best_rf_regressor,
    "XGBoost": best_xgb_regressor
}

best_model_name = comparison_df.loc[
    comparison_df["R2"].idxmax(),
    "Model"
]

best_model = model_dict[best_model_name]

print(f"\nBest Regression Model : {best_model_name}")

# ============================================================
# Create SHAP Explainer
# ============================================================

explainer = shap.TreeExplainer(best_model)

print("\nGenerating SHAP Values...")

shap_values = explainer.shap_values(X_test)

print("SHAP Values Generated Successfully.")

# ============================================================
# SHAP Summary Plot
# ============================================================

print("\nGenerating SHAP Summary Plot...")

plt.figure(figsize=(10,6))

shap.summary_plot(

    shap_values,

    X_test,

    feature_names=X_train.columns,

    show=False

)

save_figure("Figure_52_SHAP_Summary.png")

plt.show()

# ============================================================
# SHAP Bar Plot
# ============================================================

print("\nGenerating SHAP Bar Plot...")

plt.figure(figsize=(10,6))

shap.summary_plot(

    shap_values,

    X_test,

    feature_names=X_train.columns,

    plot_type="bar",

    show=False

)

save_figure("Figure_53_SHAP_Bar.png")

plt.show()

# ============================================================
# Most Important Feature
# ============================================================

if isinstance(shap_values, list):
    mean_shap = np.mean(np.abs(shap_values), axis=(0,1))
else:
    mean_shap = np.abs(shap_values).mean(axis=0)

important_feature = X_train.columns[np.argmax(mean_shap)]

print(f"\nMost Important Feature : {important_feature}")

# ============================================================
# Dependence Plot
# ============================================================

shap.dependence_plot(

    important_feature,

    shap_values,

    X_test,

    feature_names=X_train.columns,

    show=False

)

save_figure("Figure_54_SHAP_Dependence.png")

plt.show()

# ============================================================
# Waterfall Plot
# ============================================================

sample_index = 0

explanation = shap.Explanation(

    values=shap_values[sample_index],

    base_values=explainer.expected_value,

    data=X_test.iloc[sample_index],

    feature_names=X_train.columns

)

shap.plots.waterfall(

    explanation,

    show=False

)

save_figure("Figure_55_SHAP_Waterfall.png")

plt.show()

# ============================================================
# Force Plot
# ============================================================

shap.force_plot(

    explainer.expected_value,

    shap_values[sample_index],

    X_test.iloc[sample_index],

    feature_names=X_train.columns,

    matplotlib=True,

    show=False

)

save_figure("Figure_56_SHAP_Force.png")

plt.show()

# ============================================================
# SHAP Feature Importance Table
# ============================================================

shap_importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Mean |SHAP|": mean_shap

})

shap_importance = shap_importance.sort_values(

    by="Mean |SHAP|",

    ascending=False

)

print("\nSHAP Feature Importance")

display(shap_importance)

shap_importance.to_csv(

    os.path.join(

        RESULT_DIR,

        "SHAP_Feature_Importance.csv"

    ),

    index=False

)

# ============================================================
# Explain One Prediction
# ============================================================

sample_prediction = best_model.predict(

    X_test.iloc[[sample_index]]

)[0]

print("\n" + "=" * 80)
print("LOCAL PREDICTION EXPLANATION")
print("=" * 80)

print(f"Predicted Modal Price : ₹{sample_prediction:.2f}")

print("\nInput Features")

display(X_test.iloc[[sample_index]])

# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 80)
print("SECTION 14 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("""

Completed

✔ SHAP Summary Plot

✔ SHAP Bar Plot

✔ SHAP Dependence Plot

✔ SHAP Waterfall Plot

✔ SHAP Force Plot

✔ Global Explainability

✔ Local Explainability

✔ SHAP Feature Importance Table

✔ Publication Quality Figures Saved

""")

In [ ]:
# ============================================================
# SECTION 15 : WEATHER API INTEGRATION
# ============================================================

print("=" * 80)
print("SECTION 15 : WEATHER API INTEGRATION")
print("=" * 80)

# ============================================================
# Install Required Library (Run Once)
# ============================================================

# !pip install requests

# ============================================================
# Import Libraries
# ============================================================

import requests
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# ============================================================
# OpenWeather API Configuration
# ============================================================

# Create a FREE API Key:
# https://openweathermap.org/api

API_KEY = "YOUR_OPENWEATHER_API_KEY"

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

print("\nWeather API Initialized Successfully.")

# ============================================================
# Weather Fetch Function
# ============================================================

def fetch_weather(city_name):

    """
    Fetch real-time weather information from OpenWeather API.

    Parameters
    ----------
    city_name : str

    Returns
    -------
    Dictionary containing:

        Temperature

        Humidity

        Rainfall

        Pressure

        Wind Speed

        Description

    """

    parameters = {

        "q": city_name,

        "appid": API_KEY,

        "units": "metric"

    }

    try:

        response = requests.get(

            BASE_URL,

            params=parameters,

            timeout=15

        )

        response.raise_for_status()

        data = response.json()

        temperature = data["main"]["temp"]

        humidity = data["main"]["humidity"]

        pressure = data["main"]["pressure"]

        wind_speed = data["wind"]["speed"]

        description = data["weather"][0]["description"]

        rainfall = 0

        if "rain" in data:

            rainfall = data["rain"].get("1h", 0)

        weather = {

            "Temperature": temperature,

            "Humidity": humidity,

            "Rainfall": rainfall,

            "Pressure": pressure,

            "Wind Speed": wind_speed,

            "Description": description

        }

        return weather

    except requests.exceptions.HTTPError:

        print("\nInvalid City Name.")

        return None

    except requests.exceptions.ConnectionError:

        print("\nInternet Connection Error.")

        return None

    except Exception as e:

        print("\nUnexpected Error")

        print(e)

        return None

# ============================================================
# User Inputs
# ============================================================

print("\nEnter Soil Parameters")

nitrogen = float(input("Nitrogen (N) : "))

phosphorus = float(input("Phosphorus (P) : "))

potassium = float(input("Potassium (K) : "))

ph = float(input("Soil pH : "))

city = input("City Name : ")

print("\nFetching Weather Information...")

weather = fetch_weather(city)

if weather is None:

    raise Exception("Weather Data Could Not Be Retrieved.")

print("\nWeather Retrieved Successfully.")

# ============================================================
# Display Weather Information
# ============================================================

print("\nCurrent Weather Information")

print("-" * 50)

print(f"City         : {city}")

print(f"Temperature  : {weather['Temperature']} °C")

print(f"Humidity     : {weather['Humidity']} %")

print(f"Rainfall     : {weather['Rainfall']} mm")

print(f"Pressure     : {weather['Pressure']} hPa")

print(f"Wind Speed   : {weather['Wind Speed']} m/s")

print(f"Condition    : {weather['Description']}")

# ============================================================
# Create Weather DataFrame
# ============================================================

weather_df = pd.DataFrame({

    "City":[city],

    "Temperature":[weather["Temperature"]],

    "Humidity":[weather["Humidity"]],

    "Rainfall":[weather["Rainfall"]],

    "Pressure":[weather["Pressure"]],

    "Wind Speed":[weather["Wind Speed"]],

    "Weather":[weather["Description"]],

    "Timestamp":[datetime.now()]

})

print("\nWeather Data")

display(weather_df)

# ============================================================
# Save Weather Information
# ============================================================

weather_csv = os.path.join(

    RESULT_DIR,

    "Current_Weather.csv"

)

weather_df.to_csv(

    weather_csv,

    index=False

)

print("\nWeather Dataset Saved Successfully.")

print(weather_csv)

# ============================================================
# Prepare Crop Recommendation Input
# ============================================================

crop_input = pd.DataFrame({

    "N":[nitrogen],

    "P":[phosphorus],

    "K":[potassium],

    "temperature":[weather["Temperature"]],

    "humidity":[weather["Humidity"]],

    "ph":[ph],

    "rainfall":[weather["Rainfall"]]

})

print("\nCrop Recommendation Input")

display(crop_input)

# ============================================================
# Load Crop Recommendation Model
# ============================================================

print("\nLoading Crop Recommendation Model...")

crop_model = joblib.load(
    os.path.join(
        MODEL_DIR,
        "DecisionTree_Crop_Model.pkl"      # Change if your best model has another name
    )
)

print("Crop Recommendation Model Loaded Successfully.")

# ============================================================
# Predict Crop
# ============================================================

print("\nPredicting Best Crop...")

crop_prediction = crop_model.predict(crop_input)[0]

print(f"\nRecommended Crop : {crop_prediction}")

# ============================================================
# Prediction Confidence (if supported)
# ============================================================

if hasattr(crop_model, "predict_proba"):

    probabilities = crop_model.predict_proba(crop_input)[0]

    confidence = np.max(probabilities) * 100

    print(f"Prediction Confidence : {confidence:.2f}%")

    classes = crop_model.classes_

    probability_df = pd.DataFrame({

        "Crop": classes,
        "Probability (%)": probabilities * 100

    })

    probability_df = probability_df.sort_values(
        by="Probability (%)",
        ascending=False
    )

    print("\nTop Predicted Crops")

    display(probability_df.head())

# ============================================================
# Save Prediction
# ============================================================

prediction_df = pd.DataFrame({

    "City": [city],
    "N": [nitrogen],
    "P": [phosphorus],
    "K": [potassium],
    "Temperature": [weather["Temperature"]],
    "Humidity": [weather["Humidity"]],
    "pH": [ph],
    "Rainfall": [weather["Rainfall"]],
    "Recommended Crop": [crop_prediction]

})

prediction_path = os.path.join(
    RESULT_DIR,
    "Crop_Prediction.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)

print("\nCrop Prediction Saved Successfully.")

# ============================================================
# Prepare Data for Section 16
# ============================================================

print("\nPreparing data for Price Prediction...")

section16_data = {

    "recommended_crop": crop_prediction,
    "city": city,
    "weather": weather,
    "soil": {

        "N": nitrogen,
        "P": phosphorus,
        "K": potassium,
        "pH": ph

    }

}

print("Data Prepared Successfully.")

# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 80)
print("SECTION 15 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"""
Recommended Crop : {crop_prediction}

Weather Data Retrieved Successfully

Crop Recommendation Completed

Prediction Saved

""")

In [ ]:
# ============================================================
# SECTION 16 : COMPLETE AGRISENSE PIPELINE (PART 1)
# ============================================================

print("=" * 80)
print("SECTION 16 : COMPLETE AGRISENSE PIPELINE")
print("=" * 80)

# ============================================================
# Load Price Dataset
# ============================================================

print("\nLoading Price Dataset...")

price_dataset_path = "final_price_dataset.csv"      # Update if required

price_df = pd.read_csv(price_dataset_path)

print("Dataset Loaded Successfully.")

print("\nDataset Shape :", price_df.shape)

# ============================================================
# Display Basic Information
# ============================================================

print("\nColumns Present")

print(price_df.columns.tolist())

# ============================================================
# Get Predicted Crop
# ============================================================

commodity = crop_prediction

print("\nRecommended Crop :", commodity)

# ============================================================
# Search Crop Records
# ============================================================

crop_records = price_df[
    price_df["Commodity"].str.lower() == commodity.lower()
].copy()

if crop_records.empty:

    raise Exception(
        f"No Market Record Found For {commodity}"
    )

print(f"\nTotal Records Found : {len(crop_records)}")

# ============================================================
# Convert Date
# ============================================================

crop_records["date"] = pd.to_datetime(
    crop_records["date"]
)

# ============================================================
# Sort Latest Record
# ============================================================

crop_records = crop_records.sort_values(
    by="date",
    ascending=False
)

latest_record = crop_records.iloc[0]

print("\nLatest Market Record Selected.")

# ============================================================
# Display Selected Record
# ============================================================

selected_market = pd.DataFrame(
    latest_record
).T

display(selected_market)

# ============================================================
# Extract Market Information
# ============================================================

selected_apmc = latest_record["APMC"]

selected_district = latest_record["district_name"]

selected_state = latest_record["state_name"]

selected_year = int(latest_record["Year"])

selected_month = int(latest_record["Month"])

selected_arrivals = float(
    latest_record["arrivals_in_qtl"]
)

selected_min_price = float(
    latest_record["min_price"]
)

selected_max_price = float(
    latest_record["max_price"]
)

print("\nSelected Market Information")

print("--------------------------------")

print("Commodity :", commodity)

print("APMC :", selected_apmc)

print("District :", selected_district)

print("State :", selected_state)

print("Year :", selected_year)

print("Month :", selected_month)

print("Arrivals :", selected_arrivals)

print("Minimum Price :", selected_min_price)

print("Maximum Price :", selected_max_price)

# ============================================================
# Load Saved Label Encoders
# ============================================================

print("\nLoading Encoders...")

apmc_encoder = joblib.load(
    os.path.join(
        MODEL_DIR,
        "APMC_Encoder.pkl"
    )
)

commodity_encoder = joblib.load(
    os.path.join(
        MODEL_DIR,
        "Commodity_Encoder.pkl"
    )
)

district_encoder = joblib.load(
    os.path.join(
        MODEL_DIR,
        "district_name_Encoder.pkl"
    )
)

state_encoder = joblib.load(
    os.path.join(
        MODEL_DIR,
        "state_name_Encoder.pkl"
    )
)

print("Encoders Loaded Successfully.")

# ============================================================
# Encode Features
# ============================================================

encoded_apmc = apmc_encoder.transform(
    [selected_apmc]
)[0]

encoded_commodity = commodity_encoder.transform(
    [commodity]
)[0]

encoded_district = district_encoder.transform(
    [selected_district]
)[0]

encoded_state = state_encoder.transform(
    [selected_state]
)[0]

print("\nCategorical Features Encoded.")

# ============================================================
# Create Model Input
# ============================================================

price_input = pd.DataFrame({

    "APMC":[encoded_apmc],

    "Commodity":[encoded_commodity],

    "Year":[selected_year],

    "Month":[selected_month],

    "arrivals_in_qtl":[selected_arrivals],

    "min_price":[selected_min_price],

    "max_price":[selected_max_price],

    "district_name":[encoded_district],

    "state_name":[encoded_state]

})

print("\nModel Input")

display(price_input)

# ============================================================
# Save Prepared Input
# ============================================================

price_input.to_csv(

    os.path.join(
        RESULT_DIR,
        "Prepared_Price_Input.csv"
    ),

    index=False

)

print("\nPrepared Input Saved Successfully.")

print("\n" + "="*80)
print("SECTION 16 (PART 1) COMPLETED")
print("="*80)

print("""

Completed

✔ Price Dataset Loaded

✔ Recommended Crop Retrieved

✔ Market Record Selected

✔ Latest Data Extracted

✔ Label Encoders Loaded

✔ Features Encoded

✔ Price Prediction Input Prepared

Ready For Part 2

""")

# ============================================================
# SECTION 16 : COMPLETE AGRISENSE PIPELINE (PART 2)
# ============================================================

print("=" * 80)
print("SECTION 16 : COMPLETE AGRISENSE PIPELINE (PART 2)")
print("=" * 80)

# ============================================================
# Load Best Price Prediction Model
# ============================================================

print("\nLoading Price Prediction Model...")

price_model = joblib.load(
    os.path.join(
        MODEL_DIR,
        "XGBoost_Price_Model.pkl"       # Change if your model name differs
    )
)

print("Price Prediction Model Loaded Successfully.")

# ============================================================
# Predict Market Price
# ============================================================

print("\nPredicting Market Price...")

predicted_price = price_model.predict(price_input)[0]

predicted_price = round(float(predicted_price), 2)

print("Prediction Completed Successfully.")

# ============================================================
# Current Date & Time
# ============================================================

current_time = datetime.now()

prediction_date = current_time.strftime("%d-%m-%Y")

prediction_time = current_time.strftime("%H:%M:%S")

# ============================================================
# AGRISENSE RESULT DASHBOARD
# ============================================================

print("\n")
print("=" * 80)
print("🌾 AGRISENSE - SMART AGRICULTURE RECOMMENDATION SYSTEM")
print("=" * 80)

print(f"\nPrediction Date : {prediction_date}")
print(f"Prediction Time : {prediction_time}")

print("\nLOCATION INFORMATION")
print("-" * 40)

print(f"City           : {city}")
print(f"District       : {selected_district}")
print(f"State          : {selected_state}")
print(f"APMC           : {selected_apmc}")

print("\nSOIL PARAMETERS")
print("-" * 40)

print(f"Nitrogen (N)   : {nitrogen}")
print(f"Phosphorus (P) : {phosphorus}")
print(f"Potassium (K)  : {potassium}")
print(f"pH             : {ph}")

print("\nWEATHER CONDITIONS")
print("-" * 40)

print(f"Temperature    : {weather['Temperature']} °C")
print(f"Humidity       : {weather['Humidity']} %")
print(f"Rainfall       : {weather['Rainfall']} mm")
print(f"Pressure       : {weather['Pressure']} hPa")
print(f"Wind Speed     : {weather['Wind Speed']} m/s")
print(f"Condition      : {weather['Description']}")

print("\nRECOMMENDED CROP")
print("-" * 40)

print(f"Crop           : {crop_prediction}")

print("\nMARKET INFORMATION")
print("-" * 40)

print(f"Year           : {selected_year}")
print(f"Month          : {selected_month}")
print(f"Arrivals       : {selected_arrivals:.2f} Quintals")
print(f"Minimum Price  : ₹{selected_min_price:.2f}")
print(f"Maximum Price  : ₹{selected_max_price:.2f}")

print("\nPREDICTED MARKET PRICE")
print("-" * 40)

print(f"Expected Modal Price : ₹{predicted_price:.2f} per Quintal")

print("\n" + "=" * 80)
print("Prediction Completed Successfully")
print("=" * 80)

# ============================================================
# Save Final Prediction
# ============================================================

final_prediction = pd.DataFrame({

    "Prediction Date":[prediction_date],

    "Prediction Time":[prediction_time],

    "City":[city],

    "District":[selected_district],

    "State":[selected_state],

    "APMC":[selected_apmc],

    "Recommended Crop":[crop_prediction],

    "Nitrogen":[nitrogen],

    "Phosphorus":[phosphorus],

    "Potassium":[potassium],

    "pH":[ph],

    "Temperature":[weather["Temperature"]],

    "Humidity":[weather["Humidity"]],

    "Rainfall":[weather["Rainfall"]],

    "Pressure":[weather["Pressure"]],

    "Wind Speed":[weather["Wind Speed"]],

    "Arrivals (Qtl)":[selected_arrivals],

    "Minimum Price":[selected_min_price],

    "Maximum Price":[selected_max_price],

    "Predicted Modal Price":[predicted_price]

})

display(final_prediction)

# ============================================================
# Save CSV
# ============================================================

prediction_file = os.path.join(
    RESULT_DIR,
    "AGRISENSE_Final_Prediction.csv"
)

final_prediction.to_csv(
    prediction_file,
    index=False
)

print("\nPrediction Saved Successfully.")

print(prediction_file)

# ============================================================
# Optional Recommendation
# ============================================================

print("\nRECOMMENDATION")

print("-" * 40)

if predicted_price >= selected_max_price:

    recommendation = "Excellent market conditions. Selling is recommended."

elif predicted_price >= selected_min_price:

    recommendation = "Average market conditions. Consider selling."

else:

    recommendation = "Market price appears low. Waiting may be beneficial."

print(recommendation)

# ============================================================
# Final Summary
# ============================================================

print("\n" + "=" * 80)
print("SECTION 16 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("""

Completed

✔ Price Model Loaded

✔ Price Prediction Completed

✔ AGRISENSE Dashboard Generated

✔ CSV Report Saved

✔ Market Recommendation Generated

✔ End-to-End Pipeline Completed Successfully

""")

In [ ]:
# ============================================================
# SECTION 17 : SAVE MODELS & DEPLOYMENT FILES
# ============================================================

print("=" * 80)
print("SECTION 17 : SAVE MODELS & DEPLOYMENT FILES")
print("=" * 80)

import os
import json
import joblib
import pandas as pd
from datetime import datetime

# ============================================================
# Create Deployment Folder
# ============================================================

DEPLOYMENT_DIR = os.path.join(os.getcwd(), "Deployment_Files")

os.makedirs(DEPLOYMENT_DIR, exist_ok=True)

print("\nDeployment Folder Created Successfully.")

# ============================================================
# Save Crop Recommendation Model
# ============================================================

crop_model_path = os.path.join(
    DEPLOYMENT_DIR,
    "Crop_Recommendation_Model.pkl"
)

joblib.dump(best_crop_model, crop_model_path)

print("Crop Recommendation Model Saved.")

# ============================================================
# Save Price Prediction Model
# ============================================================

price_model_path = os.path.join(
    DEPLOYMENT_DIR,
    "Price_Prediction_Model.pkl"
)

joblib.dump(best_xgb_regressor, price_model_path)

print("Price Prediction Model Saved.")

# ============================================================
# Save Label Encoders
# ============================================================

encoders = {

    "Commodity_Encoder.pkl": commodity_encoder,

    "APMC_Encoder.pkl": apmc_encoder,

    "District_Encoder.pkl": district_encoder,

    "State_Encoder.pkl": state_encoder

}

print("\nSaving Label Encoders...")

for filename, encoder in encoders.items():

    joblib.dump(

        encoder,

        os.path.join(

            DEPLOYMENT_DIR,

            filename

        )

    )

print("All Encoders Saved Successfully.")

# ============================================================
# Save Feature Columns
# ============================================================

crop_features = [

    "N",

    "P",

    "K",

    "temperature",

    "humidity",

    "ph",

    "rainfall"

]

price_features = [

    "APMC",

    "Commodity",

    "Year",

    "Month",

    "arrivals_in_qtl",

    "min_price",

    "max_price",

    "district_name",

    "state_name"

]

feature_dictionary = {

    "Crop Features": crop_features,

    "Price Features": price_features

}

feature_file = os.path.join(

    DEPLOYMENT_DIR,

    "Feature_Columns.json"

)

with open(feature_file, "w") as file:

    json.dump(

        feature_dictionary,

        file,

        indent=4

    )

print("Feature Columns Saved.")

# ============================================================
# Save Model Metadata
# ============================================================

metadata = {

    "Project Name": "AGRISENSE",

    "Crop Recommendation Model": "Decision Tree",

    "Price Prediction Model": "XGBoost",

    "Crop Accuracy": float(best_crop_accuracy),

    "Price R2 Score": float(xgb_r2),

    "Created On": datetime.now().strftime("%d-%m-%Y %H:%M:%S"),

    "Developer": "Harshwardhan Sambhaji Bavale"

}

metadata_file = os.path.join(

    DEPLOYMENT_DIR,

    "Model_Metadata.json"

)

with open(metadata_file, "w") as file:

    json.dump(

        metadata,

        file,

        indent=4

    )

print("Metadata Saved.")

# ============================================================
# Save Final Prediction
# ============================================================

prediction_path = os.path.join(

    DEPLOYMENT_DIR,

    "Latest_Prediction.csv"

)

final_prediction.to_csv(

    prediction_path,

    index=False

)

print("Latest Prediction Saved.")

# ============================================================
# Save Requirements File
# ============================================================

requirements = [

    "numpy",

    "pandas",

    "scikit-learn",

    "xgboost",

    "shap",

    "matplotlib",

    "seaborn",

    "joblib",

    "requests"

]

requirements_file = os.path.join(

    DEPLOYMENT_DIR,

    "requirements.txt"

)

with open(requirements_file, "w") as file:

    for package in requirements:

        file.write(package + "\n")

print("requirements.txt Saved.")

# ============================================================
# Deployment Summary
# ============================================================

deployment_files = pd.DataFrame({

    "File Name":[

        "Crop_Recommendation_Model.pkl",

        "Price_Prediction_Model.pkl",

        "Commodity_Encoder.pkl",

        "APMC_Encoder.pkl",

        "District_Encoder.pkl",

        "State_Encoder.pkl",

        "Feature_Columns.json",

        "Model_Metadata.json",

        "Latest_Prediction.csv",

        "requirements.txt"

    ]

})

print("\nDeployment Files")

display(deployment_files)

# ============================================================
# Completion Message
# ============================================================

print("\n" + "=" * 80)
print("SECTION 17 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("""

Deployment Package Created Successfully

Files Saved

✔ Crop Recommendation Model

✔ Price Prediction Model

✔ Commodity Encoder

✔ APMC Encoder

✔ District Encoder

✔ State Encoder

✔ Feature Columns

✔ Metadata

✔ Latest Prediction

✔ Requirements File

Project Ready For Deployment

""")


In [ ]:
# ============================================================
# SECTION 18.1A : LOAD DEPLOYMENT FILES
# ============================================================

print("=" * 80)
print("SECTION 18 : AGRISENSE FINAL DEMONSTRATION")
print("=" * 80)

import os
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

# ============================================================
# Deployment Folder
# ============================================================

DEPLOYMENT_DIR = os.path.join(os.getcwd(), "Deployment_Files")

if not os.path.exists(DEPLOYMENT_DIR):
    raise FileNotFoundError("Deployment_Files folder not found!")

print("\nDeployment Folder Found.")

# ============================================================
# Load Crop Recommendation Model
# ============================================================

crop_model = joblib.load(
    os.path.join(
        DEPLOYMENT_DIR,
        "Crop_Recommendation_Model.pkl"
    )
)

print("Crop Recommendation Model Loaded.")

# ============================================================
# Load Price Prediction Model
# ============================================================

price_model = joblib.load(
    os.path.join(
        DEPLOYMENT_DIR,
        "Price_Prediction_Model.pkl"
    )
)

print("Price Prediction Model Loaded.")

# ============================================================
# Load Encoders
# ============================================================

commodity_encoder = joblib.load(
    os.path.join(
        DEPLOYMENT_DIR,
        "Commodity_Encoder.pkl"
    )
)

apmc_encoder = joblib.load(
    os.path.join(
        DEPLOYMENT_DIR,
        "APMC_Encoder.pkl"
    )
)

district_encoder = joblib.load(
    os.path.join(
        DEPLOYMENT_DIR,
        "District_Encoder.pkl"
    )
)

state_encoder = joblib.load(
    os.path.join(
        DEPLOYMENT_DIR,
        "State_Encoder.pkl"
    )
)

print("All Encoders Loaded.")

# ============================================================
# Load Feature Columns
# ============================================================

with open(
    os.path.join(
        DEPLOYMENT_DIR,
        "Feature_Columns.json"
    ),
    "r"
) as file:

    feature_columns = json.load(file)

crop_features = feature_columns["Crop Features"]
price_features = feature_columns["Price Features"]

print("Feature Columns Loaded.")

# ============================================================
# Load Metadata
# ============================================================

with open(
    os.path.join(
        DEPLOYMENT_DIR,
        "Model_Metadata.json"
    ),
    "r"
) as file:

    metadata = json.load(file)

print("\nPROJECT INFORMATION")
print("-" * 40)

for key, value in metadata.items():
    print(f"{key} : {value}")

print("-" * 40)

# ============================================================
# Load Market Dataset
# ============================================================

price_dataset = pd.read_csv("final_price_dataset.csv")

print("\nMarket Dataset Loaded.")
print("Total Records :", len(price_dataset))

# print("\nSECTION 18.1A COMPLETED SUCCESSFULLY")

# ============================================================
# SECTION 18.1B : USER INPUT & WEATHER API
# ============================================================

print("=" * 80)
print("SECTION 18.1B : USER INPUT & WEATHER DATA")
print("=" * 80)

import requests

# ============================================================
# OpenWeather API Key
# ============================================================

API_KEY = "YOUR_OPENWEATHER_API_KEY"

# ============================================================
# User Input
# ============================================================

print("\nEnter Soil Information\n")

city = input("Enter City : ")

nitrogen = float(input("Nitrogen (N) : "))
phosphorus = float(input("Phosphorus (P) : "))
potassium = float(input("Potassium (K) : "))
ph = float(input("Soil pH : "))

# ============================================================
# Weather Function
# ============================================================

def fetch_weather(city):

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

    response = requests.get(url)

    if response.status_code != 200:
        raise Exception("Unable to fetch weather data.")

    data = response.json()

    weather = {

        "temperature": data["main"]["temp"],

        "humidity": data["main"]["humidity"],

        "weather": data["weather"][0]["description"]

    }

    rainfall = 0

    if "rain" in data:

        rainfall = data["rain"].get("1h", 0)

    weather["rainfall"] = rainfall

    return weather

# ============================================================
# Fetch Weather
# ============================================================

print("\nFetching Live Weather...")

weather = fetch_weather(city)

temperature = weather["temperature"]
humidity = weather["humidity"]
rainfall = weather["rainfall"]
weather_condition = weather["weather"]

print("\nWeather Retrieved Successfully\n")

print(f"City           : {city}")
print(f"Temperature    : {temperature} °C")
print(f"Humidity       : {humidity} %")
print(f"Rainfall       : {rainfall} mm")
print(f"Condition      : {weather_condition}")

# ============================================================
# Prepare Crop Model Input
# ============================================================

crop_input = pd.DataFrame({

    "N": [nitrogen],

    "P": [phosphorus],

    "K": [potassium],

    "temperature": [temperature],

    "humidity": [humidity],

    "ph": [ph],

    "rainfall": [rainfall]

})

print("\nCrop Model Input")

display(crop_input)

print("\nSECTION 18.1B COMPLETED SUCCESSFULLY")

# ============================================================
# SECTION 18.1C : CROP RECOMMENDATION
# ============================================================

print("=" * 80)
print("SECTION 18.1C : CROP RECOMMENDATION")
print("=" * 80)

# ============================================================
# Predict Crop
# ============================================================

print("\nPredicting Best Suitable Crop...")

crop_prediction = crop_model.predict(crop_input)[0]

print("\nRecommended Crop :", crop_prediction)

# ============================================================
# Prediction Confidence
# ============================================================

try:

    probabilities = crop_model.predict_proba(crop_input)[0]

    confidence = np.max(probabilities) * 100

    print(f"Prediction Confidence : {confidence:.2f}%")

except:

    confidence = None

    print("Prediction Confidence : Not Available")

# ============================================================
# Save Prediction
# ============================================================

crop_result = pd.DataFrame({

    "City": [city],

    "Nitrogen": [nitrogen],

    "Phosphorus": [phosphorus],

    "Potassium": [potassium],

    "pH": [ph],

    "Temperature": [temperature],

    "Humidity": [humidity],

    "Rainfall": [rainfall],

    "Weather": [weather_condition],

    "Recommended Crop": [crop_prediction],

    "Confidence (%)": [
        round(confidence, 2) if confidence is not None else "N/A"
    ]

})

print("\nCrop Recommendation Result")

display(crop_result)

# ============================================================
# Save CSV
# ============================================================

crop_result.to_csv(

    os.path.join(

        DEPLOYMENT_DIR,

        "Crop_Recommendation_Result.csv"

    ),

    index=False

)

print("\nCrop Recommendation Result Saved Successfully.")

# ============================================================
# Variables for Next Section
# ============================================================

recommended_crop = crop_prediction

print("\nCrop Stored for Price Prediction.")

print("\nSECTION 18.1C COMPLETED SUCCESSFULLY")

# ============================================================
# SECTION 18.1D : MARKET PRICE PREDICTION
# ============================================================

print("=" * 80)
print("SECTION 18.1D : MARKET PRICE PREDICTION")
print("=" * 80)

# ============================================================
# Search Recommended Crop
# ============================================================

print("\nSearching Latest Market Record...")

crop_records = price_dataset[
    price_dataset["Commodity"].str.lower() == recommended_crop.lower()
].copy()

if crop_records.empty:
    raise Exception(f"No Market Data Found For {recommended_crop}")

# ============================================================
# Latest Record
# ============================================================

crop_records["date"] = pd.to_datetime(
    crop_records["Year"].astype(str) +
    "-" +
    crop_records["Month"].astype(str) +
    "-01"
)

crop_records = crop_records.sort_values(
    by="date",
    ascending=False
)

latest = crop_records.iloc[0]

print("\nLatest Record Found Successfully.")

# ============================================================
# Display Market Information
# ============================================================

print("\nMarket Information")
print("-" * 40)

print("Commodity      :", latest["Commodity"])
print("APMC           :", latest["APMC"])
print("District       :", latest["district_name"])
print("State          :", latest["state_name"])
print("Year           :", latest["Year"])
print("Month          :", latest["Month"])
print("Arrivals       :", latest["arrivals_in_qtl"])
print("Minimum Price  :", latest["min_price"])
print("Maximum Price  :", latest["max_price"])

# ============================================================
# Encode Categorical Features
# ============================================================

commodity = commodity_encoder.transform(
    [latest["Commodity"]]
)[0]

apmc = apmc_encoder.transform(
    [latest["APMC"]]
)[0]

district = district_encoder.transform(
    [latest["district_name"]]
)[0]

state = state_encoder.transform(
    [latest["state_name"]]
)[0]

# ============================================================
# Prepare Price Input
# ============================================================

price_input = pd.DataFrame({

    "APMC": [apmc],

    "Commodity": [commodity],

    "Year": [latest["Year"]],

    "Month": [latest["Month"]],

    "arrivals_in_qtl": [latest["arrivals_in_qtl"]],

    "min_price": [latest["min_price"]],

    "max_price": [latest["max_price"]],

    "district_name": [district],

    "state_name": [state]

})

print("\nPrice Model Input")

display(price_input)

# ============================================================
# Predict Market Price
# ============================================================

print("\nPredicting Expected Market Price...")

predicted_price = price_model.predict(price_input)[0]

predicted_price = round(float(predicted_price), 2)

print("\nPredicted Modal Price : ₹", predicted_price)

# ============================================================
# Store Prediction
# ============================================================

prediction_result = {

    "Crop": recommended_crop,

    "Predicted Price": predicted_price,

    "District": latest["district_name"],

    "State": latest["state_name"],

    "APMC": latest["APMC"]

}

print("\nMarket Price Prediction Completed Successfully.")

print("\nSECTION 18.1D COMPLETED SUCCESSFULLY")

# ============================================================
# SECTION 18.1E : AGRISENSE FINAL DASHBOARD
# ============================================================

print("=" * 80)
print("SECTION 18.1E : AGRISENSE DASHBOARD")
print("=" * 80)

# ============================================================
# Market Recommendation
# ============================================================

if predicted_price >= latest["max_price"]:

    market_status = "EXCELLENT"

    recommendation = "Excellent market conditions. Selling is highly recommended."

elif predicted_price >= latest["min_price"]:

    market_status = "GOOD"

    recommendation = "Good market conditions. Selling is recommended."

else:

    market_status = "LOW"

    recommendation = "Market price is relatively low. Consider waiting."

# ============================================================
# Final Dashboard
# ============================================================

print("\n")
print("=" * 80)
print("                    AGRISENSE FINAL RESULT")
print("=" * 80)

print(f"Date                 : {datetime.now().strftime('%d-%m-%Y')}")
print(f"Time                 : {datetime.now().strftime('%H:%M:%S')}")

print("\nLOCATION")
print("-" * 80)
print(f"City                 : {city}")

print("\nWEATHER")
print("-" * 80)
print(f"Temperature          : {temperature:.2f} °C")
print(f"Humidity             : {humidity:.2f} %")
print(f"Rainfall             : {rainfall:.2f} mm")
print(f"Condition            : {weather_condition}")

print("\nSOIL INFORMATION")
print("-" * 80)
print(f"Nitrogen             : {nitrogen}")
print(f"Phosphorus           : {phosphorus}")
print(f"Potassium            : {potassium}")
print(f"pH                   : {ph}")

print("\nCROP RECOMMENDATION")
print("-" * 80)
print(f"Recommended Crop     : {recommended_crop}")

if confidence is not None:
    print(f"Confidence           : {confidence:.2f}%")

print("\nMARKET DETAILS")
print("-" * 80)
print(f"District             : {latest['district_name']}")
print(f"State                : {latest['state_name']}")
print(f"APMC                 : {latest['APMC']}")
print(f"Arrivals             : {latest['arrivals_in_qtl']}")
print(f"Minimum Price        : ₹ {latest['min_price']}")
print(f"Maximum Price        : ₹ {latest['max_price']}")

print("\nPRICE PREDICTION")
print("-" * 80)
print(f"Expected Modal Price : ₹ {predicted_price:.2f} / Quintal")

print("\nMARKET STATUS")
print("-" * 80)
print(f"Status               : {market_status}")
print(f"Recommendation       : {recommendation}")

print("=" * 80)

# ============================================================
# Save Dashboard
# ============================================================

dashboard = pd.DataFrame({

    "City":[city],
    "Temperature":[temperature],
    "Humidity":[humidity],
    "Rainfall":[rainfall],
    "Weather":[weather_condition],
    "Nitrogen":[nitrogen],
    "Phosphorus":[phosphorus],
    "Potassium":[potassium],
    "pH":[ph],
    "Recommended Crop":[recommended_crop],
    "Predicted Price":[predicted_price],
    "Market Status":[market_status],
    "Recommendation":[recommendation]

})

dashboard.to_csv(

    os.path.join(
        DEPLOYMENT_DIR,
        "AGRISENSE_Dashboard.csv"
    ),

    index=False

)

print("\nDashboard Saved Successfully.")

print("\nSECTION 18 PART 1 COMPLETED SUCCESSFULLY")






In [ ]:
# ============================================================
# SECTION 18.2A : SHAP EXPLAINABILITY
# ============================================================

print("=" * 80)
print("SECTION 18.2A : SHAP EXPLAINABILITY")
print("=" * 80)

import shap
import matplotlib.pyplot as plt
import os

# ============================================================
# Create SHAP Folder
# ============================================================

SHAP_DIR = os.path.join(DEPLOYMENT_DIR, "SHAP_Results")

os.makedirs(SHAP_DIR, exist_ok=True)

print("\nSHAP Results Folder Created Successfully.")

# ============================================================
# Create SHAP Explainer
# ============================================================

print("\nCreating SHAP Explainer...")

explainer = shap.TreeExplainer(price_model)

print("SHAP Explainer Created Successfully.")

# ============================================================
# Calculate SHAP Values
# ============================================================

print("\nCalculating SHAP Values...")

shap_values = explainer.shap_values(price_input)

print("SHAP Values Calculated Successfully.")

# ============================================================
# SHAP Summary Plot
# ============================================================

print("\nGenerating SHAP Summary Plot...")

plt.figure(figsize=(10,6))

shap.summary_plot(

    shap_values,

    price_input,

    show=False

)

plt.tight_layout()

summary_path = os.path.join(

    SHAP_DIR,

    "SHAP_Summary.png"

)

plt.savefig(

    summary_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print("Summary Plot Saved.")

# ============================================================
# SHAP Feature Importance Plot
# ============================================================

print("\nGenerating SHAP Feature Importance Plot...")

plt.figure(figsize=(10,6))

shap.summary_plot(

    shap_values,

    price_input,

    plot_type="bar",

    show=False

)

plt.tight_layout()

bar_path = os.path.join(

    SHAP_DIR,

    "SHAP_Feature_Importance.png"

)

plt.savefig(

    bar_path,

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print("Feature Importance Plot Saved.")

# ============================================================
# Waterfall Plot
# ============================================================

print("\nGenerating SHAP Waterfall Plot...")

try:

    explanation = shap.Explanation(

        values=shap_values[0],

        base_values=explainer.expected_value,

        data=price_input.iloc[0],

        feature_names=price_input.columns

    )

    shap.plots.waterfall(

        explanation,

        max_display=10,

        show=False

    )

    waterfall_path = os.path.join(

        SHAP_DIR,

        "SHAP_Waterfall.png"

    )

    plt.savefig(

        waterfall_path,

        dpi=300,

        bbox_inches="tight"

    )

    plt.show()

    print("Waterfall Plot Saved.")

except Exception as e:

    print("Waterfall Plot could not be generated.")

    print(e)

# ============================================================
# SHAP Feature Contribution Table
# ============================================================

print("\nCreating SHAP Contribution Table...")

contribution_df = pd.DataFrame({

    "Feature": price_input.columns,

    "SHAP Value": shap_values[0]

})

contribution_df["Absolute Value"] = contribution_df["SHAP Value"].abs()

contribution_df = contribution_df.sort_values(

    by="Absolute Value",

    ascending=False

)

display(contribution_df)

contribution_df.to_csv(

    os.path.join(

        SHAP_DIR,

        "SHAP_Contribution_Table.csv"

    ),

    index=False

)

print("Contribution Table Saved.")

# ============================================================
# SHAP Result Summary
# ============================================================

print("\n" + "="*80)
print("SHAP EXPLAINABILITY COMPLETED")
print("="*80)

print("""

Generated Files

✔ SHAP_Summary.png

✔ SHAP_Feature_Importance.png

✔ SHAP_Waterfall.png

✔ SHAP_Contribution_Table.csv

""")

print("SECTION 18.2A COMPLETED SUCCESSFULLY")

# ============================================================
# SECTION 18.2B : FINAL REPORT & PROJECT COMPLETION
# ============================================================

print("=" * 80)
print("SECTION 18.2B : FINAL REPORT")
print("=" * 80)

from datetime import datetime
import os

# ============================================================
# Create Final Report
# ============================================================

print("\nGenerating Final AGRISENSE Report...")

final_report = pd.DataFrame({

    "Prediction Date":[datetime.now().strftime("%d-%m-%Y")],

    "Prediction Time":[datetime.now().strftime("%H:%M:%S")],

    "City":[city],

    "District":[latest["district_name"]],

    "State":[latest["state_name"]],

    "APMC":[latest["APMC"]],

    "Temperature (°C)":[temperature],

    "Humidity (%)":[humidity],

    "Rainfall (mm)":[rainfall],

    "Weather":[weather_condition],

    "Nitrogen":[nitrogen],

    "Phosphorus":[phosphorus],

    "Potassium":[potassium],

    "Soil pH":[ph],

    "Recommended Crop":[recommended_crop],

    "Predicted Modal Price":[predicted_price],

    "Market Status":[market_status],

    "Recommendation":[recommendation]

})

display(final_report)

# ============================================================
# Save Final Report
# ============================================================

report_path = os.path.join(

    DEPLOYMENT_DIR,

    "AGRISENSE_Final_Report.csv"

)

final_report.to_csv(

    report_path,

    index=False

)

print("\nFinal Report Saved Successfully.")

# ============================================================
# Prediction History
# ============================================================

history_path = os.path.join(

    DEPLOYMENT_DIR,

    "Prediction_History.csv"

)

if os.path.exists(history_path):

    history = pd.read_csv(history_path)

    history = pd.concat(

        [history, final_report],

        ignore_index=True

    )

else:

    history = final_report.copy()

history.to_csv(

    history_path,

    index=False

)

print("Prediction History Updated Successfully.")

# ============================================================
# Display Saved Files
# ============================================================

saved_files = pd.DataFrame({

    "Generated Files":[

        "Crop_Recommendation_Result.csv",

        "AGRISENSE_Dashboard.csv",

        "AGRISENSE_Final_Report.csv",

        "Prediction_History.csv",

        "SHAP_Contribution_Table.csv",

        "SHAP_Summary.png",

        "SHAP_Feature_Importance.png",

        "SHAP_Waterfall.png"

    ]

})

print("\nGenerated Files")

display(saved_files)

# ============================================================
# Project Statistics
# ============================================================

print("\n")
print("=" * 80)
print("PROJECT SUMMARY")
print("=" * 80)

print(f"Project Name            : AGRISENSE")

print(f"Developer               : Harshwardhan Sambhaji Bavale")

print(f"Prediction Date         : {datetime.now().strftime('%d-%m-%Y')}")

print(f"Prediction Time         : {datetime.now().strftime('%H:%M:%S')}")

print(f"Recommended Crop        : {recommended_crop}")

print(f"Predicted Market Price  : ₹ {predicted_price:.2f}")

print(f"Market Status           : {market_status}")

print(f"Recommendation          : {recommendation}")

print(f"Location                : {city}")

print(f"Weather                 : {weather_condition}")

print("=" * 80)

# ============================================================
# Final Success Message
# ============================================================

print("""

████████████████████████████████████████████████████████████████████

                    AGRISENSE PROJECT COMPLETED

████████████████████████████████████████████████████████████████████

End-to-End Pipeline Executed Successfully

✔ Weather API Integration

✔ Crop Recommendation

✔ Market Dataset Search

✔ Price Prediction

✔ Explainable AI (SHAP)

✔ Dashboard Generation

✔ Final Report Generation

✔ Prediction History Saved

✔ Deployment Files Ready

Project Status : SUCCESS

Thank you for using AGRISENSE.

████████████████████████████████████████████████████████████████████

""")

print("=" * 80)
print("NOTEBOOK EXECUTED SUCCESSFULLY")
print("=" * 80)